In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/csv2006/finalcsv/final_chembl.csv


In [4]:
d="/kaggle/input/datasets/csv2006/finalcsv/final_chembl.csv"
data=pd.read_csv(d)
data.columns.tolist()

/tmp/ipykernel_49/635113452.py:2: DtypeWarning: Columns (12,13,16,20) have mixed types. Specify dtype option on import or set low_memory=False.
  data=pd.read_csv(d)


['activity_id',
 'assay_id',
 'doc_id',
 'compound_chembl_id',
 'standard_inchi_key',
 'inchikey_14',
 'canonical_smiles',
 'uniprot',
 'target_name',
 'target_organism',
 'standard_type',
 'standard_value',
 'standard_units',
 'standard_relation',
 'pchembl_value',
 'label_type',
 'assay_call',
 'assay_type',
 'confidence_score',
 'species_mismatch',
 'data_validity_comment',
 'potential_duplicate']

In [1]:
import pandas as pd, numpy as np
CO    = "/kaggle/input/datasets/csv2006/coconut-csv/coconut_csv-10-2026.csv"
FINAL = "/kaggle/input/datasets/csv2006/finalcsv/final_chembl.csv"        # the 22-column file
OUT   = "/kaggle/working/final_chembl_np.csv"
LOOK  = "/kaggle/working/coconut_lookup.csv"

co = pd.read_csv(CO, usecols=["standard_inchi_key","collections","np_likeness"], low_memory=False)
co["inchikey_14"] = co.standard_inchi_key.str[:14]
co = co.dropna(subset=["inchikey_14"])

EXCL = {"DrugBankNP","ChEMBL NPs","NCI DTP data","FooDB","Exposome-explorer","Phenol-explorer",
        "InterBioScreen Ltd","Specs Natural Products","AnalytiCon Discovery NPs","Indofine Chemical Company",
        "PhytoLab phyproof® Reference Substances","PubChem NPs","ChEBI NPs","ChemSpider NPs",
        "Wikidata Natural Products","ZINC NP"}
def union(s):
    out = set()
    for v in s.dropna(): out.update(v.split("|"))
    out.discard(""); return out
agg = co.groupby("inchikey_14").agg(coll_set=("collections", union), npl=("np_likeness","max"))
agg["n_coll"] = agg.coll_set.map(len)
agg["n_core"] = agg.coll_set.map(lambda s: len(s - EXCL))
lookup = agg[["n_coll","n_core","npl"]].reset_index()
lookup.to_csv(LOOK, index=False); print("lookup:", lookup.shape)

f = pd.read_csv(FINAL, low_memory=False); n0 = len(f)
if "inchikey_14" not in f.columns: f["inchikey_14"] = f.standard_inchi_key.str[:14]
f = f.merge(lookup, on="inchikey_14", how="left"); assert len(f) == n0
f["np_tier"] = np.select([f.inchikey_14.isna(), f.n_coll.isna(), f.n_coll>=10, f.n_coll>=5, f.n_coll>=2],
                         ["no_structure","unmatched","np_10plus","np_5to9","np_2to4"], default="np_1")
f["human"]      = f.target_organism.eq("Homo sapiens")
f["strict"]     = f.label_type.eq("exact") & f.assay_type.eq("B") & f.human
f["drugmatrix"] = f.doc_id.eq(59306)
f.to_csv(OUT, index=False)
print(f.shape); print(f.np_tier.value_counts())
print("exact rows with pchembl:", f[f.label_type=="exact"].pchembl_value.notna().mean())

lookup: (479690, 4)
(2591981, 29)
np_tier
unmatched       2306834
np_2to4           95848
np_1              76113
np_5to9           53859
np_10plus         48959
no_structure      10368
Name: count, dtype: int64
exact rows with pchembl: 0.9761292266206226


In [19]:
import os
import pandas as pd

directories = ["/kaggle/input", "/kaggle/working"]

for dir_path in directories:
    print(f"\n{'='*50}")
    print(f"DIRECTORY: {dir_path}")
    print(f"{'='*50}")
    
    if not os.path.exists(dir_path):
        print("Directory does not exist.")
        continue

    for root, _, files in os.walk(dir_path):
        for file in sorted(files):
            if file.endswith(".csv"):
                full_path = os.path.join(root, file)
                try:
                    # Read only 2 rows for speed and low memory usage
                    df = pd.read_csv(full_path, nrows=2, low_memory=False)
                    rel_path = os.path.relpath(full_path, dir_path)
                    print(f"\nFile: {rel_path}")
                    print(f"Full Path: {full_path}")
                    print(f"Column Count: {len(df.columns)}")
                    print("Columns:")
                    for idx, col in enumerate(df.columns, 1):
                        print(f"  {idx:2d}. {col}")
                except Exception as e:
                    print(f"\nCould not read {file}: {e}")


DIRECTORY: /kaggle/input

File: datasets/csv2006/finalcsv/final_chembl.csv
Full Path: /kaggle/input/datasets/csv2006/finalcsv/final_chembl.csv
Column Count: 22
Columns:
   1. activity_id
   2. assay_id
   3. doc_id
   4. compound_chembl_id
   5. standard_inchi_key
   6. inchikey_14
   7. canonical_smiles
   8. uniprot
   9. target_name
  10. target_organism
  11. standard_type
  12. standard_value
  13. standard_units
  14. standard_relation
  15. pchembl_value
  16. label_type
  17. assay_call
  18. assay_type
  19. confidence_score
  20. species_mismatch
  21. data_validity_comment
  22. potential_duplicate

File: datasets/csv2006/chembl37-fileb-csv/chembl37_fileB.csv
Full Path: /kaggle/input/datasets/csv2006/chembl37-fileb-csv/chembl37_fileB.csv
Column Count: 26
Columns:
   1. activity_id
   2. assay_id
   3. doc_id
   4. molregno
   5. standard_relation
   6. standard_value
   7. standard_units
   8. standard_type
   9. standard_text_value
  10. activity_comment
  11. data_validit

In [9]:
import os, pandas as pd

# 1. List all files saved in /kaggle/working/
print("Files in working dir:", os.listdir("/kaggle/working/"))

# 2. Check the columns and shape of your main dataset
df = pd.read_csv("/kaggle/working/final_chembl_np.csv", nrows=5)
print("Column count:", len(df.columns))
print("Columns:", df.columns.tolist())

Files in working dir: ['final_chembl_np.csv', 'family_summary.csv', 'family_summary1.csv', 'coconut_lookup.csv', 'family_summary3.csv', '.virtual_documents', 'protein_family_lookup.csv']
Column count: 30
Columns: ['activity_id', 'assay_id', 'doc_id', 'compound_chembl_id', 'standard_inchi_key', 'inchikey_14', 'canonical_smiles', 'uniprot', 'target_name', 'target_organism', 'standard_type', 'standard_value', 'standard_units', 'standard_relation', 'pchembl_value', 'label_type', 'assay_call', 'assay_type', 'confidence_score', 'species_mismatch', 'data_validity_comment', 'potential_duplicate', 'n_coll', 'n_core', 'npl', 'np_tier', 'human', 'strict', 'drugmatrix', 'family']


In [ ]:
import pandas as pd, numpy as np, requests, io, time, re
OUT = "/kaggle/working/final_chembl_np.csv"
f = pd.read_csv(OUT, low_memory=False)
print(f.shape)                       # expect (2591981, 29)
NP_COL, NP_T = "n_core", 5           # natural-like = n_core >= 5 (change here)

# 1) UniProt family lookup
accs = sorted(f.uniprot.dropna().unique().tolist()); print("proteins:", len(accs))
parts = []
for i in range(0, len(accs), 100):
    chunk = accs[i:i+100]
    for attempt in range(3):
        r = requests.get("https://rest.uniprot.org/uniprotkb/accessions",
            params={"accessions": ",".join(chunk), "fields": "accession,protein_families,ec,protein_name",
                    "format": "tsv"}, timeout=60)
        if r.status_code == 200: break
        time.sleep(3)
    else:
        print("batch failed", i, r.status_code, r.text[:200]); continue
    t = pd.read_csv(io.StringIO(r.text), sep="\t", dtype=str)
    t.columns = ["uniprot","families_raw","ec","protein_names"][:t.shape[1]]
    parts.append(t)
prot = pd.concat(parts, ignore_index=True).drop_duplicates("uniprot")
print("returned:", len(prot), "of", len(accs))

def broad(fam, ec, nm_):
    fam = fam.lower() if isinstance(fam, str) else ""
    ec  = ec if isinstance(ec, str) else ""
    if "g-protein coupled receptor" in fam: return "GPCR"
    if "nuclear hormone receptor" in fam: return "Nuclear receptor"
    if "kinase" in fam or re.search(r"\b2\.7\.(1|10|11|12)\.", ec): return "Kinase"
    if "channel" in fam or "ligand-gated" in fam or "cys-loop" in fam: return "Ion channel"
    if "cytochrome p450" in fam: return "Cytochrome P450"
    if re.search(r"\b3\.4\.", ec) or "peptidase" in fam or "protease" in fam: return "Protease"
    if re.search(r"\b3\.1\.3\.", ec) or "phosphatase" in fam: return "Phosphatase"
    if any(k in fam for k in ["bromodomain","bet family","deacetylase","sirtuin","histone","demethylase","methyltransferase"]): return "Epigenetic"
    if "transporter" in fam or "solute carrier" in fam: return "Transporter"
    if ec: return "Other enzyme"
    if fam: return "Other family"
    return "No family annotation"
prot["family"] = [broad(a, b, c) for a, b, c in zip(prot.families_raw, prot.ec, prot.protein_names)]

cnt = f.groupby("uniprot").agg(rows=("activity_id","size"), compounds=("inchikey_14","nunique")).reset_index()
prot = pd.DataFrame({"uniprot": accs}).merge(prot, on="uniprot", how="left").merge(cnt, on="uniprot", how="left")
prot["family"] = prot.family.fillna("Unmapped")
prot.to_csv("/kaggle/working/protein_family_lookup.csv", index=False)
print(prot.family.value_counts())
print(prot.groupby("family").rows.sum().sort_values(ascending=False))
print(prot[prot.family.isin(["Other family","Other enzyme"])].families_raw.fillna("").str[:60].value_counts().head(25))

f = f.merge(prot[["uniprot","family"]], on="uniprot", how="left")
f.to_csv(OUT, index=False)

# 2) family summary
def summarize(sub, label):
    sub = sub[sub.np_tier != "no_structure"].copy()
    sub["side"] = np.where(sub[NP_COL] >= NP_T, "np", np.where(sub.n_coll.isna(), "other", "grey"))
    fam = sub.groupby("family")
    out = pd.DataFrame({
        "proteins":        fam.uniprot.nunique(),
        "rows_total":      fam.size(),
        "rows_np":         sub[sub.side=="np"].groupby("family").size(),
        "rows_other":      sub[sub.side=="other"].groupby("family").size(),
        "compounds_np":    sub[sub.side=="np"].groupby("family").inchikey_14.nunique(),
        "compounds_other": sub[sub.side=="other"].groupby("family").inchikey_14.nunique()}).fillna(0)
    pc = (sub[sub.side.isin(["np","other"])].groupby(["uniprot","side"]).inchikey_14.nunique()
          .unstack(fill_value=0))
    for c in ["np","other"]:
        if c not in pc: pc[c] = 0
    pc = pc.join(prot.set_index("uniprot").family)
    pc["both"] = (pc.np > 0) & (pc.other > 0); pc["both3"] = (pc.np >= 3) & (pc.other >= 3)
    out["proteins_np"]        = pc[pc.np > 0].groupby("family").size()
    out["proteins_other"]     = pc[pc.other > 0].groupby("family").size()
    out["proteins_both"]      = pc.groupby("family").both.sum()
    out["proteins_both_ge3"]  = pc.groupby("family").both3.sum()
    out = out.fillna(0).astype(int)
    out["np_row_share"] = (out.rows_np / (out.rows_np + out.rows_other)).round(4)
    out.insert(0, "row_set", label)
    return out.reset_index()

summ = pd.concat([summarize(f[~f.drugmatrix], "all_noDM"),
                  summarize(f[f.strict & ~f.drugmatrix], "strict_noDM")], ignore_index=True)
summ.to_csv("/kaggle/working/family_summary1.csv", index=False)
for rs in ["all_noDM","strict_noDM"]:
    print(rs); print(summ[summ.row_set==rs].sort_values("rows_total", ascending=False).to_string(index=False))

In [ ]:
import pandas as pd, numpy as np, requests, io, time, re
OUT = "/kaggle/working/final_chembl_np.csv"
f = pd.read_csv(OUT, low_memory=False)
print(f.shape)                       # expect (2591981, 29)
NP_COL, NP_T = "n_core", 5           # natural-like = n_core >= 5 (change here)

# 1) UniProt family lookup
accs = sorted(f.uniprot.dropna().unique().tolist()); print("proteins:", len(accs))
parts = []
for i in range(0, len(accs), 100):
    chunk = accs[i:i+100]
    for attempt in range(3):
        r = requests.get("https://rest.uniprot.org/uniprotkb/accessions",
            params={"accessions": ",".join(chunk), "fields": "accession,protein_families,ec,protein_name",
                    "format": "tsv"}, timeout=60)
        if r.status_code == 200: break
        time.sleep(3)
    else:
        print("batch failed", i, r.status_code, r.text[:200]); continue
    t = pd.read_csv(io.StringIO(r.text), sep="\t", dtype=str)
    t.columns = ["uniprot","families_raw","ec","protein_names"][:t.shape[1]]
    parts.append(t)
prot = pd.concat(parts, ignore_index=True).drop_duplicates("uniprot")
print("returned:", len(prot), "of", len(accs))

def broad(fam, ec, nm_):
    fam = fam.lower() if isinstance(fam, str) else ""
    ec  = ec if isinstance(ec, str) else ""
    if "g-protein coupled receptor" in fam: return "GPCR"
    if "nuclear hormone receptor" in fam: return "Nuclear receptor"
    if "kinase" in fam or re.search(r"\b2\.7\.(1|10|11|12)\.", ec): return "Kinase"
    if "channel" in fam or "ligand-gated" in fam or "cys-loop" in fam: return "Ion channel"
    if "cytochrome p450" in fam: return "Cytochrome P450"
    if re.search(r"\b3\.4\.", ec) or "peptidase" in fam or "protease" in fam: return "Protease"
    if re.search(r"\b3\.1\.3\.", ec) or "phosphatase" in fam: return "Phosphatase"
    if any(k in fam for k in ["bromodomain","bet family","deacetylase","sirtuin","histone","demethylase","methyltransferase"]): return "Epigenetic"
    if "transporter" in fam or "solute carrier" in fam: return "Transporter"
    if ec: return "Other enzyme"
    if fam: return "Other family"
    return "No family annotation"
prot["family"] = [broad(a, b, c) for a, b, c in zip(prot.families_raw, prot.ec, prot.protein_names)]

cnt = f.groupby("uniprot").agg(rows=("activity_id","size"), compounds=("inchikey_14","nunique")).reset_index()
prot = pd.DataFrame({"uniprot": accs}).merge(prot, on="uniprot", how="left").merge(cnt, on="uniprot", how="left")
prot["family"] = prot.family.fillna("Unmapped")
prot.to_csv("/kaggle/working/protein_family_lookup.csv", index=False)
print(prot.family.value_counts())
print(prot.groupby("family").rows.sum().sort_values(ascending=False))
print(prot[prot.family.isin(["Other family","Other enzyme"])].families_raw.fillna("").str[:60].value_counts().head(25))

f = f.drop(columns=["family"], errors="ignore")
f.to_csv(OUT, index=False)

# 2) family summary
def summarize(sub, label):
    sub = sub[sub.np_tier != "no_structure"].copy()
    sub["side"] = np.where(sub[NP_COL] >= NP_T, "np", np.where(sub.n_coll.isna(), "other", "grey"))
    fam = sub.groupby("family")
    out = pd.DataFrame({
        "proteins":        fam.uniprot.nunique(),
        "rows_total":      fam.size(),
        "rows_np":         sub[sub.side=="np"].groupby("family").size(),
        "rows_other":      sub[sub.side=="other"].groupby("family").size(),
        "compounds_np":    sub[sub.side=="np"].groupby("family").inchikey_14.nunique(),
        "compounds_other": sub[sub.side=="other"].groupby("family").inchikey_14.nunique()}).fillna(0)
    pc = (sub[sub.side.isin(["np","other"])].groupby(["uniprot","side"]).inchikey_14.nunique()
          .unstack(fill_value=0))
    for c in ["np","other"]:
        if c not in pc: pc[c] = 0
    pc = pc.join(prot.set_index("uniprot").family)
    pc["both"] = (pc.np > 0) & (pc.other > 0); pc["both3"] = (pc.np >= 3) & (pc.other >= 3)
    out["proteins_np"]        = pc[pc.np > 0].groupby("family").size()
    out["proteins_other"]     = pc[pc.other > 0].groupby("family").size()
    out["proteins_both"]      = pc.groupby("family").both.sum()
    out["proteins_both_ge3"]  = pc.groupby("family").both3.sum()
    out = out.fillna(0).astype(int)
    out["np_row_share"] = (out.rows_np / (out.rows_np + out.rows_other)).round(4)
    out.insert(0, "row_set", label)
    return out.reset_index()

summ = pd.concat([summarize(f[~f.drugmatrix], "all_noDM"),
                  summarize(f[f.strict & ~f.drugmatrix], "strict_noDM")], ignore_index=True)
summ.to_csv("/kaggle/working/family_summary2.csv", index=False)
for rs in ["all_noDM","strict_noDM"]:
    print(rs); print(summ[summ.row_set==rs].sort_values("rows_total", ascending=False).to_string(index=False))

In [ ]:
import pandas as pd, numpy as np, requests, io, time, re
OUT = "/kaggle/working/final_chembl_np.csv"
f = pd.read_csv(OUT, low_memory=False)
print(f.shape)                       # expect (2591981, 29)
NP_COL, NP_T = "n_core", 5           # natural-like = n_core >= 5 (change here)

# 1) UniProt family lookup
accs = sorted(f.uniprot.dropna().unique().tolist()); print("proteins:", len(accs))
parts = []
for i in range(0, len(accs), 100):
    chunk = accs[i:i+100]
    for attempt in range(3):
        r = requests.get("https://rest.uniprot.org/uniprotkb/accessions",
            params={"accessions": ",".join(chunk), "fields": "accession,protein_families,ec,protein_name",
                    "format": "tsv"}, timeout=60)
        if r.status_code == 200: break
        time.sleep(3)
    else:
        print("batch failed", i, r.status_code, r.text[:200]); continue
    t = pd.read_csv(io.StringIO(r.text), sep="\t", dtype=str)
    t.columns = ["uniprot","families_raw","ec","protein_names"][:t.shape[1]]
    parts.append(t)
prot = pd.concat(parts, ignore_index=True).drop_duplicates("uniprot")
print("returned:", len(prot), "of", len(accs))

def broad(fam, ec, nm_):
    fam = fam.lower() if isinstance(fam, str) else ""
    ec  = ec if isinstance(ec, str) else ""
    if "g-protein coupled receptor" in fam: return "GPCR"
    if "nuclear hormone receptor" in fam: return "Nuclear receptor"
    if "kinase" in fam or re.search(r"\b2\.7\.(1|10|11|12)\.", ec): return "Kinase"
    if "channel" in fam or "ligand-gated" in fam or "cys-loop" in fam: return "Ion channel"
    if "cytochrome p450" in fam: return "Cytochrome P450"
    if re.search(r"\b3\.4\.", ec) or "peptidase" in fam or "protease" in fam: return "Protease"
    if re.search(r"\b3\.1\.3\.", ec) or "phosphatase" in fam: return "Phosphatase"
    if any(k in fam for k in ["bromodomain","bet family","deacetylase","sirtuin","histone","demethylase","methyltransferase"]): return "Epigenetic"
    if "transporter" in fam or "solute carrier" in fam: return "Transporter"
    if ec: return "Other enzyme"
    if fam: return "Other family"
    return "No family annotation"
prot["family"] = [broad(a, b, c) for a, b, c in zip(prot.families_raw, prot.ec, prot.protein_names)]

cnt = f.groupby("uniprot").agg(rows=("activity_id","size"), compounds=("inchikey_14","nunique")).reset_index()
prot = pd.DataFrame({"uniprot": accs}).merge(prot, on="uniprot", how="left").merge(cnt, on="uniprot", how="left")
prot["family"] = prot.family.fillna("Unmapped")
prot.to_csv("/kaggle/working/protein_family_lookup.csv", index=False)
print(prot.family.value_counts())
print(prot.groupby("family").rows.sum().sort_values(ascending=False))
print(prot[prot.family.isin(["Other family","Other enzyme"])].families_raw.fillna("").str[:60].value_counts().head(25))
f = f.drop(columns=["family"], errors="ignore")
f = f.merge(prot[["uniprot", "family"]], on="uniprot", how="left") 
f.to_csv(OUT, index=False)

# 2) family summary
def summarize(sub, label):
    sub = sub[sub.np_tier != "no_structure"].copy()
    sub["side"] = np.where(sub[NP_COL] >= NP_T, "np", np.where(sub.n_coll.isna(), "other", "grey"))
    fam = sub.groupby("family")
    out = pd.DataFrame({
        "proteins":        fam.uniprot.nunique(),
        "rows_total":      fam.size(),
        "rows_np":         sub[sub.side=="np"].groupby("family").size(),
        "rows_other":      sub[sub.side=="other"].groupby("family").size(),
        "compounds_np":    sub[sub.side=="np"].groupby("family").inchikey_14.nunique(),
        "compounds_other": sub[sub.side=="other"].groupby("family").inchikey_14.nunique()}).fillna(0)
    pc = (sub[sub.side.isin(["np","other"])].groupby(["uniprot","side"]).inchikey_14.nunique()
          .unstack(fill_value=0))
    for c in ["np","other"]:
        if c not in pc: pc[c] = 0
    pc = pc.join(prot.set_index("uniprot").family)
    pc["both"] = (pc.np > 0) & (pc.other > 0); pc["both3"] = (pc.np >= 3) & (pc.other >= 3)
    out["proteins_np"]        = pc[pc.np > 0].groupby("family").size()
    out["proteins_other"]     = pc[pc.other > 0].groupby("family").size()
    out["proteins_both"]      = pc.groupby("family").both.sum()
    out["proteins_both_ge3"]  = pc.groupby("family").both3.sum()
    out = out.fillna(0).astype(int)
    out["np_row_share"] = (out.rows_np / (out.rows_np + out.rows_other)).round(4)
    out.insert(0, "row_set", label)
    return out.reset_index()

summ = pd.concat([summarize(f[~f.drugmatrix], "all_noDM"),
                  summarize(f[f.strict & ~f.drugmatrix], "strict_noDM")], ignore_index=True)
summ.to_csv("/kaggle/working/family_summary3.csv", index=False)
for rs in ["all_noDM","strict_noDM"]:
    print(rs); print(summ[summ.row_set==rs].sort_values("rows_total", ascending=False).to_string(index=False))

In [14]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.5/29.5 MB 33.9 MB/s  0:00:00m0:00:0100:01


In [ ]:
"""
Baseline (from scratch): compound fingerprint + protein sequence features -> pActivity.
Question: does a model trained on synthetic-like compounds do worse on natural-like compounds,
especially on proteins it has never seen?

Works directly on your Kaggle file /kaggle/working/final_chembl_np.csv (no sequences.csv needed:
sequences are downloaded from UniProt the first time and cached).
Turn Kaggle "Internet" ON for the first run.
Needs: rdkit, scikit-learn, pandas, numpy, scipy, requests (all present on Kaggle except maybe rdkit:
    !pip install rdkit    in a cell if import fails)

Run:  python baseline_np_transfer_v2.py        (or paste into a notebook cell and call main())
First run it with QUICK_TEST = True to check everything works, then set it to False.
"""
import os, re, time
import numpy as np, pandas as pd
from itertools import product
from scipy.stats import pearsonr, spearmanr
from sklearn.ensemble import HistGradientBoostingRegressor
from rdkit import Chem, RDLogger
from rdkit.Chem import rdFingerprintGenerator

RDLogger.DisableLog("rdApp.*")

# =============================== CONFIG ===============================
CSV        = "/kaggle/working/final_chembl_np.csv"
SEQ_CACHE  = "/kaggle/working/sequences.csv"          # created automatically
OUT_CSV    = "/kaggle/working/baseline_results.csv"

COL_SMILES = "canonical_smiles"
COL_PROT   = "uniprot"
COL_DOC    = "doc_id"
COL_NCOLL  = "n_coll"          # COCONUT collection count (empty/NaN = not in COCONUT)
COL_FAMILY = "family"          # optional: per-family report (set to None to skip)

NP_MIN_COLL = 1                # natural-like if n_coll >= this (try 5 for a stricter label)
# Optional row filters. Set a name to None to skip. They keep rows where the column is True/1.
# CHECK what these columns mean before trusting them (print value_counts first):
KEEP_IF_TRUE   = ["strict"]    # e.g. ["strict", "human"]
DROP_IF_TRUE   = ["drugmatrix"]
EXCLUDE_DOCS   = {59306}       # DrugMatrix panel (also removed by the drugmatrix column)

MAX_PER_DOC_FRAC = 0.10        # no single document may supply more than 10% of rows
MAX_SYNTH_ROWS   = 120000      # subsample synthetic-like rows (keeps memory/time OK); natural-like rows all kept
QUICK_TEST       = False        # True = tiny run just to check the code works
QUICK_ROWS       = 20000
TEST_PROT_FRAC   = 0.20        # fraction of proteins held out ("new proteins")
SEEDS            = [0,1,2]         # e.g. [0, 1, 2] for mean +/- sd
MIN_FAMILY_N     = 30          # per-family lines only if at least this many test rows
# ======================================================================

AA = "ACDEFGHIKLMNPQRSTVWY"
DIPEP = {a + b: i for i, (a, b) in enumerate(product(AA, AA))}
_FPGEN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)


def truthy(s):
    return s.astype(str).str.strip().str.lower().isin(["true", "1", "1.0", "yes", "t"])


def clean_uniprot(x):
    """ChEMBL sometimes stores several accessions in one cell; keep the first."""
    x = str(x).strip()
    return re.split(r"[|,;\s]+", x)[0] if x and x.lower() != "nan" else ""


# ---------------------------- features ----------------------------
def prot_feats(seq):
    seq = "".join(c for c in str(seq).upper() if c in AA)
    comp, dip = np.zeros(20), np.zeros(400)
    if len(seq) > 1:
        for c in seq:
            comp[AA.index(c)] += 1
        comp /= len(seq)
        for i in range(len(seq) - 1):
            dip[DIPEP[seq[i:i + 2]]] += 1
        dip /= len(seq) - 1
    return np.concatenate([comp, dip, [np.log1p(len(seq))]]).astype(np.float32)


def fp(smi):
    m = Chem.MolFromSmiles(str(smi))
    return None if m is None else _FPGEN.GetFingerprintAsNumPy(m).astype(np.uint8)


# ---------------------------- UniProt sequences ----------------------------
def parse_uniprot_tsv(text):
    rows = []
    for line in text.strip().splitlines()[1:]:      # skip header
        parts = line.split("\t")
        if len(parts) >= 2 and parts[0] and parts[1]:
            rows.append((parts[0], parts[1]))
    return rows


def fetch_sequences(accessions, cache=None, batch=100, sleep=0.3):
    """Download sequences from UniProt (needs internet). Cached in `cache` so it only runs once."""
    import requests
    cache = cache or SEQ_CACHE
    have = {}
    if os.path.exists(cache):
        c = pd.read_csv(cache)
        have = dict(zip(c["uniprot"], c["sequence"]))
    todo = [a for a in accessions if a not in have]
    print(f"sequences: {len(have)} cached, {len(todo)} to download")
    for i in range(0, len(todo), batch):
        chunk = todo[i:i + batch]
        query = " OR ".join(f"accession:{a}" for a in chunk)
        params = {"query": query, "format": "tsv", "fields": "accession,sequence", "size": 500}
        for attempt in range(3):
            try:
                r = requests.get("https://rest.uniprot.org/uniprotkb/search", params=params, timeout=60)
                r.raise_for_status()
                for acc, s in parse_uniprot_tsv(r.text):
                    have[acc] = s
                break
            except Exception as e:                       # network trouble: retry, then give up on this batch
                print(f"  batch {i//batch} attempt {attempt+1} failed: {e}")
                time.sleep(2)
        if (i // batch) % 10 == 0:
            print(f"  downloaded batch {i//batch + 1} of {(len(todo)-1)//batch + 1}")
        time.sleep(sleep)
    pd.DataFrame({"uniprot": list(have), "sequence": list(have.values())}).to_csv(cache, index=False)
    return have


# ---------------------------- data prep ----------------------------
def build_table(path=None):
    df = pd.read_csv(path or CSV, low_memory=False)
    print("rows loaded:", len(df))

    # pActivity: exact values only; pchembl if present, else from nM value
    df = df[df["standard_relation"].astype(str).str.strip() == "="].copy()
    pv = pd.to_numeric(df["pchembl_value"], errors="coerce")
    val = pd.to_numeric(df["standard_value"], errors="coerce")
    nm = df["standard_units"].astype(str).str.strip().eq("nM") & (val > 0)
    fallback = 9 - np.log10(val.where(nm))
    both = pv.notna() & fallback.notna()
    if both.sum() > 100:
        print("pchembl vs computed: mean abs diff =", float(np.abs(pv[both] - fallback[both]).mean()))
    df["pvalue"] = pv.fillna(fallback)
    df = df.dropna(subset=["pvalue", COL_SMILES, COL_PROT])

    for c in (KEEP_IF_TRUE or []):
        if c in df.columns:
            print(f"filter keep {c}: {truthy(df[c]).sum()} of {len(df)} rows are True")
            df = df[truthy(df[c])]
        else:
            print(f"WARNING column {c} not found; skipping filter")
    for c in (DROP_IF_TRUE or []):
        if c in df.columns:
            df = df[~truthy(df[c])]
    df = df[~df[COL_DOC].isin(EXCLUDE_DOCS)]

    df[COL_PROT] = df[COL_PROT].map(clean_uniprot)
    df = df[df[COL_PROT] != ""]
    keep = [COL_SMILES, COL_PROT, "pvalue", COL_DOC, COL_NCOLL] + ([COL_FAMILY] if COL_FAMILY in df.columns else [])
    # one median value per compound-protein pair (removes duplicates)
    agg = {"pvalue": "median", COL_DOC: "first", COL_NCOLL: "max"}
    if COL_FAMILY in df.columns:
        agg[COL_FAMILY] = "first"
    df = df[keep].groupby([COL_SMILES, COL_PROT], as_index=False).agg(agg)
    df["is_np"] = df[COL_NCOLL].fillna(0) >= NP_MIN_COLL
    print(f"after cleaning: {len(df):,} pairs | natural-like {df.is_np.sum():,} | other {(~df.is_np).sum():,}"
          f" | proteins {df[COL_PROT].nunique():,}")
    return df


def cap_docs(df, frac, seed):
    limit = max(1, int(len(df) * frac))
    parts = [g.sample(limit, random_state=seed) if len(g) > limit else g for _, g in df.groupby(COL_DOC)]
    return pd.concat(parts)


def report(name, y, p, rows):
    if len(y) < 10:
        print(f"{name:<44} n={len(y):>6}  (too few rows)")
        return
    r, s = pearsonr(y, p)[0], spearmanr(y, p)[0]
    rmse = float(np.sqrt(np.mean((y - p) ** 2)))
    print(f"{name:<44} n={len(y):>6}  Pearson={r:6.3f}  Spearman={s:6.3f}  RMSE={rmse:5.2f}")
    rows.append(dict(setting=name, n=len(y), pearson=r, spearman=s, rmse=rmse))


# ---------------------------- main ----------------------------
def main():
    df = build_table()
    df = cap_docs(df, MAX_PER_DOC_FRAC, seed=0)

    # shrink to a size that fits Kaggle memory: keep ALL natural-like rows, subsample the rest
    n_syn_cap = QUICK_ROWS if QUICK_TEST else MAX_SYNTH_ROWS
    np_df, syn_df = df[df.is_np], df[~df.is_np]
    if len(syn_df) > n_syn_cap:
        syn_df = syn_df.sample(n_syn_cap, random_state=0)
    if QUICK_TEST and len(np_df) > QUICK_ROWS // 4:
        np_df = np_df.sample(QUICK_ROWS // 4, random_state=0)
    df = pd.concat([np_df, syn_df]).reset_index(drop=True)
    print(f"working set: {len(df):,} pairs (natural-like {df.is_np.sum():,})")

    # protein sequences (downloaded once, then cached)
    seqs = fetch_sequences(sorted(df[COL_PROT].unique()))
    df = df[df[COL_PROT].isin(seqs)].reset_index(drop=True)
    print(f"pairs with a sequence: {len(df):,} | proteins: {df[COL_PROT].nunique():,}")

    # features
    fps = {s: fp(s) for s in df[COL_SMILES].unique()}
    df = df[df[COL_SMILES].map(lambda s: fps[s] is not None)].reset_index(drop=True)
    pf = {u: prot_feats(seqs[u]) for u in df[COL_PROT].unique()}
    X = np.hstack([np.stack(df[COL_SMILES].map(fps).values),
                   np.stack(df[COL_PROT].map(pf).values)]).astype(np.float32)
    y = df["pvalue"].values.astype(np.float32)
    syn, nat = ~df.is_np.values, df.is_np.values
    print("feature matrix:", X.shape)

    rows = []
    for seed in SEEDS:
        rng = np.random.RandomState(seed)
        prots = np.array(sorted(df[COL_PROT].unique()))
        rng.shuffle(prots)
        test_prots = set(prots[: int(len(prots) * TEST_PROT_FRAC)])
        in_test = df[COL_PROT].isin(test_prots).values

        seen_syn = (~in_test) & syn
        hold = seen_syn & (rng.rand(len(df)) < 0.10)
        train = seen_syn & ~hold
        print(f"\n--- seed {seed}: train rows {train.sum():,} | held-out proteins {len(test_prots)} ---")

        model = HistGradientBoostingRegressor(max_iter=100 if QUICK_TEST else 300,
                                              learning_rate=0.08, random_state=seed)
        model.fit(X[train], y[train])

        sets = {
            "synthetic-like, seen proteins": hold,
            "synthetic-like, NEW proteins": in_test & syn,
            "natural-like, seen proteins": (~in_test) & nat,
            "natural-like, NEW proteins": in_test & nat,
        }
        pred = {k: model.predict(X[m]) if m.sum() else np.array([]) for k, m in sets.items()}
        for k, m in sets.items():
            report(f"[seed {seed}] {k}", y[m], pred[k], rows)

        if COL_FAMILY in df.columns:
            print("  per-family, natural-like rows (seen + new proteins together):")
            m_all = nat & ~train
            p_all = model.predict(X[m_all]) if m_all.sum() else np.array([])
            sub = df[m_all].assign(pred=p_all)
            for fam, g in sub.groupby(COL_FAMILY):
                if len(g) >= MIN_FAMILY_N:
                    print(f"    {str(fam):<22} n={len(g):>5}  Spearman={spearmanr(g.pvalue, g.pred)[0]:6.3f}")

    res = pd.DataFrame(rows)
    if len(res):
        res.to_csv(OUT_CSV, index=False)
        if len(SEEDS) > 1:
            print("\nmean +/- sd across seeds:")
            print(res.groupby(res.setting.str.replace(r"^\[seed \d+\] ", "", regex=True))
                     [["pearson", "spearman", "rmse"]].agg(["mean", "std"]).round(3))
        print("\nsaved:", OUT_CSV)


if __name__ == "__main__":
    main()


In [17]:
import pandas as pd

# Load main dataset
df = pd.read_csv("/kaggle/working/final_chembl_np.csv", low_memory=False)

print("--- 1. VALUE COUNTS ---")
for col in ["np_tier", "strict", "human", "drugmatrix"]:
    if col in df.columns:
        print(f"\nValue counts for '{col}':")
        print(df[col].value_counts(dropna=False))
    else:
        print(f"\nColumn '{col}' not found.")

print("\n--- 2. SAMPLE ROWS ---")
show_cols = [c for c in ["inchikey_14", "np_tier", "n_coll", "n_core", "npl", "family"] if c in df.columns]
print(df[show_cols].head(10).to_string(index=False))

print("\n--- 3. UNMATCHED COMPOUNDS CHECK ---")
if "np_tier" in df.columns and "n_coll" in df.columns:
    unmatched = df[df["np_tier"] == "unmatched"]
    print("n_coll values for unmatched compounds:")
    print(unmatched["n_coll"].value_counts(dropna=False).head(5))

--- 1. VALUE COUNTS ---

Value counts for 'np_tier':
np_tier
unmatched       151555
np_1              3963
np_2to4           3510
np_5to9           1861
np_10plus         1774
no_structure        77
Name: count, dtype: int64

Value counts for 'strict':
strict
False    91234
True     71506
Name: count, dtype: int64

Value counts for 'human':
human
True     88361
False    74379
Name: count, dtype: int64

Value counts for 'drugmatrix':
drugmatrix
False    162740
Name: count, dtype: int64

--- 2. SAMPLE ROWS ---
   inchikey_14   np_tier  n_coll  n_core  npl          family
BEBACPIIZGRKGG unmatched     NaN     NaN  NaN    Other enzyme
SUKVIELCKKEBOJ unmatched     NaN     NaN  NaN    Other enzyme
ZFJHZUAZBGPPQK unmatched     NaN     NaN  NaN    Other enzyme
LFSFDIDSIQKNDC unmatched     NaN     NaN  NaN    Other enzyme
LCCPHXWRAPZDSN unmatched     NaN     NaN  NaN    Other enzyme
SZRMSHBRXJWBFG unmatched     NaN     NaN  NaN    Other enzyme
SZRMSHBRXJWBFG unmatched     NaN     NaN  NaN Cytoch

In [20]:
import pandas as pd
df = pd.read_csv("/kaggle/working/final_chembl_np.csv", low_memory=False)
for c in ["np_tier","strict","human","drugmatrix","potential_duplicate","data_validity_comment","standard_type"]:
    print(df[c].value_counts(dropna=False).head(8), "\n")

np_tier
unmatched       151555
np_1              3963
np_2to4           3510
np_5to9           1861
np_10plus         1774
no_structure        77
Name: count, dtype: int64 

strict
False    91234
True     71506
Name: count, dtype: int64 

human
True     88361
False    74379
Name: count, dtype: int64 

drugmatrix
False    162740
Name: count, dtype: int64 

potential_duplicate
0    156364
1      6376
Name: count, dtype: int64 

data_validity_comment
NaN                              154721
Outside typical range              7957
Potential transcription error        44
Potential author error               18
Name: count, dtype: int64 

standard_type
IC50    91521
Ki      67310
Kd       3909
Name: count, dtype: int64 



In [21]:
import pandas as pd, numpy as np
df = pd.read_csv("/kaggle/working/final_chembl_np.csv", low_memory=False)
print(df.standard_relation.value_counts(dropna=False), "\n")
print(pd.crosstab(df.strict, df.standard_relation), "\n")
print(pd.crosstab(df.strict, df.np_tier), "\n")
print(pd.crosstab(df.human, df.strict), "\n")
print(df.doc_id.value_counts().head(5), "\n")

def t(s): return s.astype(str).str.strip().str.lower().isin(["true","1","1.0"])
m = (t(df.human) & (df.standard_relation.astype(str).str.strip() == "=")
     & (df.potential_duplicate == 0) & df.data_validity_comment.isna()
     & (df.np_tier != "no_structure"))
d = df[m]
d_u = d.uniprot.astype(str).str.split(r"[|,;\s]+").str[0]
print("usable rows:", len(d), "| proteins:", d_u.nunique())
print("natural-like rows:", (d.np_tier != "unmatched").sum())
print("proteins with natural-like rows:", d_u[d.np_tier != "unmatched"].nunique())

standard_relation
=      143591
>       17633
<        1180
~         137
>>         80
>=         75
NaN        35
<<          5
<=          4
Name: count, dtype: int64 

standard_relation     <  <<  <=      =      >  >=  >>    ~
strict                                                    
False              1180   5   4  72085  17633  75  80  137
True                  0   0   0  71506      0   0   0    0 

np_tier  no_structure  np_1  np_10plus  np_2to4  np_5to9  unmatched
strict                                                             
False              56  2211       1076     2043     1093      84755
True               21  1752        698     1467      768      66800 

strict  False  True 
human               
False   74379      0
True    16855  71506 

doc_id
15725    447
17331    354
15632    328
12131    327
6647     319
Name: count, dtype: int64 

usable rows: 71001 | proteins: 647
natural-like rows: 4079
proteins with natural-like rows: 422


In [23]:
import io
import re
import time
import requests
import numpy as np
import pandas as pd
from itertools import product
from scipy.stats import pearsonr, spearmanr
from sklearn.ensemble import HistGradientBoostingRegressor

from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import AllChem

RDLogger.DisableLog("rdApp.*")

# ----------------------------------------------------------------------
# 1. DATA PREPARATION & FILTERING
# ----------------------------------------------------------------------
OUT = "/kaggle/working/final_chembl_np.csv"
SEQ_CSV = "/kaggle/working/sequences.csv"
SEED = [42,43,44]

df = pd.read_csv(OUT, low_memory=False)

# Exact filtering rules
def to_bool(s):
    return s.astype(str).str.strip().str.lower().isin(["true", "1", "1.0"])

mask = (
    to_bool(df["human"]) &
    (df["standard_relation"].astype(str).str.strip() == "=") &
    (df["potential_duplicate"] == 0) &
    df["data_validity_comment"].isna() &
    (df["np_tier"] != "no_structure")
)
df = df[mask].copy()

# Keep first UniProt ID
df["uniprot"] = df["uniprot"].astype(str).str.split(r"[|,;\s]+").str[0]

# Calculate pvalue (pchembl_value or 9 - log10(standard_value) in nM)
def calc_pvalue(row):
    p = row.get("pchembl_value")
    if pd.notna(p):
        try:
            return float(p)
        except ValueError:
            pass
    val = row.get("standard_value")
    unit = str(row.get("standard_units")).strip().lower()
    if pd.notna(val) and unit in ["nm", "nanomolar"]:
        try:
            v = float(val)
            if v > 0:
                return 9.0 - np.log10(v)
        except ValueError:
            pass
    return np.nan

df["pvalue"] = [calc_pvalue(row) for _, row in df.iterrows()]
df = df.dropna(subset=["pvalue", "canonical_smiles", "uniprot"]).copy()

# Aggregate duplicates to median value per compound-protein pair
df = (
    df.groupby(["canonical_smiles", "uniprot"], as_index=False)
      .agg({
          "pvalue": "median",
          "np_tier": "first",
          "doc_id": "first"
      })
)

print(f"Usable aggregated pairs: {len(df):,} | Unique proteins: {df['uniprot'].nunique():,}")

# ----------------------------------------------------------------------
# 2. UNIPROT SEQUENCE FETCHING
# ----------------------------------------------------------------------
accs = sorted(df["uniprot"].unique().tolist())
try:
    seq_df = pd.read_csv(SEQ_CSV)
    missing_accs = [a for a in accs if a not in seq_df["uniprot"].values]
except FileNotFoundError:
    seq_df = pd.DataFrame(columns=["uniprot", "sequence"])
    missing_accs = accs

if missing_accs:
    print(f"Fetching sequences for {len(missing_accs):,} proteins from UniProt...")
    parts = []
    for i in range(0, len(missing_accs), 100):
        chunk = missing_accs[i:i+100]
        for attempt in range(3):
            r = requests.get(
                "https://rest.uniprot.org/uniprotkb/accessions",
                params={"accessions": ",".join(chunk), "fields": "accession,sequence", "format": "tsv"},
                timeout=60
            )
            if r.status_code == 200:
                break
            time.sleep(2)
        else:
            continue
        t = pd.read_csv(io.StringIO(r.text), sep="\t", dtype=str)
        t.columns = ["uniprot", "sequence"]
        parts.append(t)
    if parts:
        new_seqs = pd.concat(parts, ignore_index=True).drop_duplicates("uniprot")
        seq_df = pd.concat([seq_df, new_seqs], ignore_index=True).drop_duplicates("uniprot")
        seq_df.to_csv(SEQ_CSV, index=False)

seq_dict = seq_df.set_index("uniprot")["sequence"].to_dict()
df = df[df["uniprot"].isin(seq_dict.keys())].copy()

# ----------------------------------------------------------------------
# 3. FEATURE EXTRACTION & SCAFFOLD COMPUTATION
# ----------------------------------------------------------------------
AA = "ACDEFGHIKLMNPQRSTVWY"
DIPEP = {a + b: i for i, (a, b) in enumerate(product(AA, AA))}

def get_prot_feats(seq):
    seq = "".join(c for c in str(seq).upper() if c in AA)
    comp = np.zeros(20)
    dip = np.zeros(400)
    if len(seq) > 1:
        for c in seq:
            comp[AA.index(c)] += 1
        comp /= len(seq)
        for i in range(len(seq) - 1):
            dip[DIPEP[seq[i:i + 2]]] += 1
        dip /= len(seq) - 1
    return np.concatenate([comp, dip, [np.log1p(len(seq))]])

def get_fp(smi, n=2048):
    m = Chem.MolFromSmiles(smi)
    if m is None:
        return None
    v = AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=n)
    return np.array(list(v), dtype=np.uint8)

def get_scaffold(smi):
    m = Chem.MolFromSmiles(smi)
    if m is None:
        return smi
    try:
        scaf = MurckoScaffold.MurckoScaffoldSmiles(mol=m)
        return scaf if scaf else smi
    except Exception:
        return smi

# Cache fingerprints, protein features, and scaffolds
unique_smiles = df["canonical_smiles"].unique()
fp_dict = {s: get_fp(s) for s in unique_smiles}
scaf_dict = {s: get_scaffold(s) for s in unique_smiles}

df = df[df["canonical_smiles"].map(lambda s: fp_dict[s] is not None)].copy()
df["scaffold"] = df["canonical_smiles"].map(scaf_dict)

unique_prots = df["uniprot"].unique()
pf_dict = {u: get_prot_feats(seq_dict[u]) for u in unique_prots}

X_fps = np.stack(df["canonical_smiles"].map(fp_dict).values)
X_pfs = np.stack(df["uniprot"].map(pf_dict).values)
X_all = np.hstack([X_fps, X_pfs]).astype(np.float32)
y_all = df["pvalue"].values.astype(np.float32)
prot_arr = df["uniprot"].values

# ----------------------------------------------------------------------
# 4. EVALUATION HELPERS
# ----------------------------------------------------------------------
def per_protein_spearman(y_true, y_pred, prots, min_samples=5):
    eval_df = pd.DataFrame({"y_true": y_true, "y_pred": y_pred, "prot": prots})
    spearmans = []
    for _, group in eval_df.groupby("prot"):
        if len(group) >= min_samples and group["y_true"].nunique() > 1:
            s = spearmanr(group["y_true"], group["y_pred"])[0]
            if not np.isnan(s):
                spearmans.append(s)
    return (np.mean(spearmans), len(spearmans)) if spearmans else (np.nan, 0)

def evaluate_set(name, mask, y_true, y_pred, prots):
    n_samples = mask.sum()
    if n_samples < 10:
        print(f"{name:<42} n={n_samples:>5}  (Too few rows to evaluate)")
        return
    
    yt, yp, pr = y_true[mask], y_pred[mask], prots[mask]
    p_r = pearsonr(yt, yp)[0]
    s_r = spearmanr(yt, yp)[0]
    rmse = np.sqrt(np.mean((yt - yp) ** 2))
    mean_sp, n_prots = per_protein_spearman(yt, yp, pr, min_samples=5)
    
    print(f"{name:<42} n={n_samples:>5} | Pooled Pearson={p_r:6.3f} | Pooled Spearman={s_r:6.3f} | Per-Prot Spearman (n>={5})={mean_sp:6.3f} ({n_prots} prots) | RMSE={rmse:5.2f}")

# ----------------------------------------------------------------------
# 5. EXPERIMENT RUNNER
# ----------------------------------------------------------------------
def run_experiment(tier_label, is_np_mask, drop_mask=None):
    ...
    drop = np.zeros(len(df), bool) if drop_mask is None else drop_mask
    syn_mask = (~is_np_mask) & (~drop)
    np_mask = is_np_mask
    print(f"\n=======================================================")
    print(f" EXPERIMENT RUN: {tier_label.upper()} NATURAL-LIKE DEFINITION")
    print(f"=======================================================")
    
    rng = np.random.RandomState(SEED)
    
    # 1. Protein-level split (20% new proteins held out)
    all_prots = np.array(sorted(df["uniprot"].unique()))
    rng.shuffle(all_prots)
    n_test_prots = int(len(all_prots) * 0.20)
    test_prots = set(all_prots[:n_test_prots])
    is_new_prot = df["uniprot"].isin(test_prots).values
    
    syn_mask = ~is_np_mask
    np_mask = is_np_mask
    
    # 2. Scaffold holdout for synthetic compounds (20% held-out scaffolds)
    syn_scaffolds = np.array(sorted(df[syn_mask]["scaffold"].unique()))
    rng.shuffle(syn_scaffolds)
    n_test_scafs = int(len(syn_scaffolds) * 0.20)
    test_scaffolds = set(syn_scaffolds[:n_test_scafs])
    is_new_scaffold = df["scaffold"].isin(test_scaffolds).values
    
    # Training set: Synthetic compounds, SEEN proteins, SEEN scaffolds
    train_mask = syn_mask & (~is_new_prot) & (~is_new_scaffold)
    
    # Test sets:
    test_syn_seen_prot_new_scaf = syn_mask & (~is_new_prot) & is_new_scaffold
    test_syn_new_prot = syn_mask & is_new_prot & is_new_scaffold
    test_np_seen_prot = np_mask & (~is_new_prot)
    test_np_new_prot = np_mask & is_new_prot
    
    # Required print of natural-like test counts before showing scores
    n_np_seen = test_np_seen_prot.sum()
    n_np_new = test_np_new_prot.sum()
    n_np_total = n_np_seen + n_np_new
    
    print(f"Train set size (Synthetic only, seen proteins & scaffolds): {train_mask.sum():,}")
    print(f"Total Natural-like Test Rows: {n_np_total:,} (Seen Proteins: {n_np_seen:,} | New Proteins: {n_np_new:,})")
    print("-----------------------------------------------------------------------")
    
    # Train GBDT model
    model = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.08, random_state=SEED)
    model.fit(X_all[train_mask], y_all[train_mask])
    
    y_preds = model.predict(X_all)
    
    # Report Results
    evaluate_set("Synthetic (Seen Prots, Unseen Scaffolds)", test_syn_seen_prot_new_scaf, y_all, y_preds, prot_arr)
    evaluate_set("Synthetic (NEW Prots / Cold Start)", test_syn_new_prot, y_all, y_preds, prot_arr)
    evaluate_set("Natural-like (Seen Prots)", test_np_seen_prot, y_all, y_preds, prot_arr)
    evaluate_set("Natural-like (NEW Prots / Cold Start)", test_np_new_prot, y_all, y_preds, prot_arr)

# ----------------------------------------------------------------------
# 6. EXECUTE BOTH RUNS
# ----------------------------------------------------------------------
# Loose definition: np_tier != "unmatched"
is_np_loose = (df["np_tier"] != "unmatched").values
run_experiment("Loose (np_tier != unmatched)", is_np_loose)

# Strict definition: np_tier in ["np_5to9", "np_10plus"]
is_np_strict = df["np_tier"].isin(["np_5to9", "np_10plus"]).values
drop_mid = df["np_tier"].isin(["np_1", "np_2to4"]).values
run_experiment("Strict (np_5to9 or np_10plus)", is_np_strict, drop_mid)
run_experiment("Strict (np_5to9 or np_10plus)", is_np_strict)

Usable aggregated pairs: 62,906 | Unique proteins: 647

 EXPERIMENT RUN: LOOSE (NP_TIER != UNMATCHED) NATURAL-LIKE DEFINITION
Train set size (Synthetic only, seen proteins & scaffolds): 38,164
Total Natural-like Test Rows: 2,702 (Seen Proteins: 2,150 | New Proteins: 552)
-----------------------------------------------------------------------
Synthetic (Seen Prots, Unseen Scaffolds)   n=10162 | Pooled Pearson= 0.705 | Pooled Spearman= 0.695 | Per-Prot Spearman (n>=5)= 0.378 (239 prots) | RMSE= 0.95
Synthetic (NEW Prots / Cold Start)         n= 2517 | Pooled Pearson= 0.238 | Pooled Spearman= 0.227 | Per-Prot Spearman (n>=5)= 0.200 (56 prots) | RMSE= 1.26
Natural-like (Seen Prots)                  n= 2150 | Pooled Pearson= 0.566 | Pooled Spearman= 0.554 | Per-Prot Spearman (n>=5)= 0.350 (136 prots) | RMSE= 1.24
Natural-like (NEW Prots / Cold Start)      n=  552 | Pooled Pearson= 0.440 | Pooled Spearman= 0.450 | Per-Prot Spearman (n>=5)= 0.180 (34 prots) | RMSE= 1.36

 EXPERIMENT RUN: STRI

In [24]:
import io
import time
import requests
import numpy as np
import pandas as pd
from itertools import product
from scipy.stats import pearsonr, spearmanr
from sklearn.ensemble import HistGradientBoostingRegressor

from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import AllChem

RDLogger.DisableLog("rdApp.*")

# ----------------------------------------------------------------------
# 1. DATA PREPARATION & FILTERING
# ----------------------------------------------------------------------
OUT = "/kaggle/working/final_chembl_np.csv"
SEQ_CSV = "/kaggle/working/sequences.csv"
SEEDS = [42, 43, 44]

df = pd.read_csv(OUT, low_memory=False)

def to_bool(s):
    return s.astype(str).str.strip().str.lower().isin(["true", "1", "1.0"])

mask = (
    to_bool(df["human"]) &
    (df["standard_relation"].astype(str).str.strip() == "=") &
    (df["potential_duplicate"] == 0) &
    df["data_validity_comment"].isna() &
    (df["np_tier"] != "no_structure")
)
df = df[mask].copy()

# Keep first UniProt ID
df["uniprot"] = df["uniprot"].astype(str).str.split(r"[|,;\s]+").str[0]

# Calculate pvalue (pchembl_value or 9 - log10(standard_value) in nM)
def calc_pvalue(row):
    p = row.get("pchembl_value")
    if pd.notna(p):
        try:
            return float(p)
        except ValueError:
            pass
    val = row.get("standard_value")
    unit = str(row.get("standard_units")).strip().lower()
    if pd.notna(val) and unit in ["nm", "nanomolar"]:
        try:
            v = float(val)
            if v > 0:
                return 9.0 - np.log10(v)
        except ValueError:
            pass
    return np.nan

df["pvalue"] = [calc_pvalue(row) for _, row in df.iterrows()]
df = df.dropna(subset=["pvalue", "canonical_smiles", "uniprot"]).copy()

# Aggregate duplicates to median value per compound-protein pair
df = (
    df.groupby(["canonical_smiles", "uniprot"], as_index=False)
      .agg({"pvalue": "median", "np_tier": "first", "doc_id": "first"})
)

print(f"Usable aggregated pairs: {len(df):,} | Unique proteins: {df['uniprot'].nunique():,}")

# ----------------------------------------------------------------------
# 2. SEQUENCES & FEATURE EXTRACTION
# ----------------------------------------------------------------------
accs = sorted(df["uniprot"].unique().tolist())
try:
    seq_df = pd.read_csv(SEQ_CSV)
    missing_accs = [a for a in accs if a not in seq_df["uniprot"].values]
except FileNotFoundError:
    seq_df = pd.DataFrame(columns=["uniprot", "sequence"])
    missing_accs = accs

if missing_accs:
    print(f"Fetching sequences for {len(missing_accs):,} proteins from UniProt...")
    parts = []
    for i in range(0, len(missing_accs), 100):
        chunk = missing_accs[i:i+100]
        for attempt in range(3):
            r = requests.get(
                "https://rest.uniprot.org/uniprotkb/accessions",
                params={"accessions": ",".join(chunk), "fields": "accession,sequence", "format": "tsv"},
                timeout=60
            )
            if r.status_code == 200:
                break
            time.sleep(2)
        else:
            continue
        t = pd.read_csv(io.StringIO(r.text), sep="\t", dtype=str)
        t.columns = ["uniprot", "sequence"]
        parts.append(t)
    if parts:
        new_seqs = pd.concat(parts, ignore_index=True).drop_duplicates("uniprot")
        seq_df = pd.concat([seq_df, new_seqs], ignore_index=True).drop_duplicates("uniprot")
        seq_df.to_csv(SEQ_CSV, index=False)

seq_dict = seq_df.set_index("uniprot")["sequence"].to_dict()
df = df[df["uniprot"].isin(seq_dict.keys())].copy()

AA = "ACDEFGHIKLMNPQRSTVWY"
DIPEP = {a + b: i for i, (a, b) in enumerate(product(AA, AA))}

def get_prot_feats(seq):
    seq = "".join(c for c in str(seq).upper() if c in AA)
    comp, dip = np.zeros(20), np.zeros(400)
    if len(seq) > 1:
        for c in seq: comp[AA.index(c)] += 1
        comp /= len(seq)
        for i in range(len(seq) - 1): dip[DIPEP[seq[i:i + 2]]] += 1
        dip /= len(seq) - 1
    return np.concatenate([comp, dip, [np.log1p(len(seq))]])

def get_fp(smi, n=2048):
    m = Chem.MolFromSmiles(smi)
    if m is None: return None
    v = AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=n)
    return np.array(list(v), dtype=np.uint8)

def get_scaffold(smi):
    m = Chem.MolFromSmiles(smi)
    if m is None: return smi
    try:
        scaf = MurckoScaffold.MurckoScaffoldSmiles(mol=m)
        return scaf if scaf else smi
    except Exception:
        return smi

unique_smiles = df["canonical_smiles"].unique()
fp_dict = {s: get_fp(s) for s in unique_smiles}
scaf_dict = {s: get_scaffold(s) for s in unique_smiles}

df = df[df["canonical_smiles"].map(lambda s: fp_dict[s] is not None)].copy()
df["scaffold"] = df["canonical_smiles"].map(scaf_dict)

unique_prots = df["uniprot"].unique()
pf_dict = {u: get_prot_feats(seq_dict[u]) for u in unique_prots}

X_fps = np.stack(df["canonical_smiles"].map(fp_dict).values)
X_pfs = np.stack(df["uniprot"].map(pf_dict).values)
X_all = np.hstack([X_fps, X_pfs]).astype(np.float32)
y_all = df["pvalue"].values.astype(np.float32)
prot_arr = df["uniprot"].values

# ----------------------------------------------------------------------
# 3. METRICS HELPERS
# ----------------------------------------------------------------------
def per_protein_spearman(y_true, y_pred, prots, min_samples=5):
    eval_df = pd.DataFrame({"y_true": y_true, "y_pred": y_pred, "prot": prots})
    spearmans = []
    for _, group in eval_df.groupby("prot"):
        if len(group) >= min_samples and group["y_true"].nunique() > 1:
            s = spearmanr(group["y_true"], group["y_pred"])[0]
            if not np.isnan(s):
                spearmans.append(s)
    return np.mean(spearmans) if spearmans else np.nan

def eval_mask(mask, y_true, y_pred, prots):
    if mask.sum() < 10:
        return np.nan, np.nan, np.nan, np.nan
    yt, yp, pr = y_true[mask], y_pred[mask], prots[mask]
    p_r = pearsonr(yt, yp)[0]
    s_r = spearmanr(yt, yp)[0]
    rmse = np.sqrt(np.mean((yt - yp) ** 2))
    pp_sp = per_protein_spearman(yt, yp, pr, min_samples=5)
    return p_r, s_r, pp_sp, rmse

# ----------------------------------------------------------------------
# 4. MULTI-SEED EXPERIMENT RUNNER
# ----------------------------------------------------------------------
def run_multi_seed(tier_label, is_np_mask, drop_mask=None):
    print(f"\n=======================================================")
    print(f" MULTI-SEED RUN (AVERAGED OVER SEEDS {SEEDS}): {tier_label.upper()}")
    print(f"=======================================================")
    
    drop = np.zeros(len(df), bool) if drop_mask is None else drop_mask
    syn_mask = (~is_np_mask) & (~drop)
    np_mask = is_np_mask
    
    set_names = [
        "Synthetic (Seen Prots, Unseen Scaffolds)",
        "Synthetic (NEW Prots / Cold Start)",
        "Natural-like (Seen Prots)",
        "Natural-like (NEW Prots / Cold Start)"
    ]
    
    results = {name: {"pearson": [], "spearman": [], "pp_spearman": [], "rmse": [], "n": []} for name in set_names}
    
    for seed in SEEDS:
        rng = np.random.RandomState(seed)
        
        # 1. Protein split (20% held out)
        all_prots = np.array(sorted(df["uniprot"].unique()))
        rng.shuffle(all_prots)
        test_prots = set(all_prots[:int(len(all_prots) * 0.20)])
        is_new_prot = df["uniprot"].isin(test_prots).values
        
        # 2. Scaffold split for synthetic compounds (20% held out)
        syn_scaffolds = np.array(sorted(df[syn_mask]["scaffold"].unique()))
        rng.shuffle(syn_scaffolds)
        test_scaffolds = set(syn_scaffolds[:int(len(syn_scaffolds) * 0.20)])
        is_new_scaffold = df["scaffold"].isin(test_scaffolds).values
        
        # Training set
        train_mask = syn_mask & (~is_new_prot) & (~is_new_scaffold)
        
        # Test sets
        masks = {
            "Synthetic (Seen Prots, Unseen Scaffolds)": syn_mask & (~is_new_prot) & is_new_scaffold,
            "Synthetic (NEW Prots / Cold Start)": syn_mask & is_new_prot & is_new_scaffold,
            "Natural-like (Seen Prots)": np_mask & (~is_new_prot),
            "Natural-like (NEW Prots / Cold Start)": np_mask & is_new_prot
        }
        
        model = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.08, random_state=seed)
        model.fit(X_all[train_mask], y_all[train_mask])
        y_preds = model.predict(X_all)
        
        for name, m in masks.items():
            pr, sr, ppsr, rmse = eval_mask(m, y_all, y_preds, prot_arr)
            results[name]["pearson"].append(pr)
            results[name]["spearman"].append(sr)
            results[name]["pp_spearman"].append(ppsr)
            results[name]["rmse"].append(rmse)
            results[name]["n"].append(m.sum())

    print(f"{'Test Subset':<42} | {'Mean n':>6} | {'Pearson':>7} | {'Spearman':>8} | {'Per-Prot Sp':>11} | {'RMSE':>5}")
    print("-" * 90)
    for name in set_names:
        n_m = np.mean(results[name]["n"])
        pr_m = np.nanmean(results[name]["pearson"])
        sr_m = np.nanmean(results[name]["spearman"])
        ppsr_m = np.nanmean(results[name]["pp_spearman"])
        rmse_m = np.nanmean(results[name]["rmse"])
        print(f"{name:<42} | {n_m:>6.0f} | {pr_m:>7.3f} | {sr_m:>8.3f} | {ppsr_m:>11.3f} | {rmse_m:>5.2f}")

# ----------------------------------------------------------------------
# 5. EXECUTE BOTH RUNS
# ----------------------------------------------------------------------
# Loose definition: np_tier != "unmatched"
is_np_loose = (df["np_tier"] != "unmatched").values
run_multi_seed("Loose (np_tier != unmatched)", is_np_loose)

# Strict definition: np_tier in ["np_5to9", "np_10plus"], dropping np_1 and np_2to4
is_np_strict = df["np_tier"].isin(["np_5to9", "np_10plus"]).values
drop_mid = df["np_tier"].isin(["np_1", "np_2to4"]).values
run_multi_seed("Strict (np_5to9 or np_10plus)", is_np_strict, drop_mid)

Usable aggregated pairs: 62,906 | Unique proteins: 647

 MULTI-SEED RUN (AVERAGED OVER SEEDS [42, 43, 44]): LOOSE (NP_TIER != UNMATCHED)
Test Subset                                | Mean n | Pearson | Spearman | Per-Prot Sp |  RMSE
------------------------------------------------------------------------------------------
Synthetic (Seen Prots, Unseen Scaffolds)   |   9448 |   0.712 |    0.704 |       0.389 |  0.93
Synthetic (NEW Prots / Cold Start)         |   2515 |   0.234 |    0.226 |       0.154 |  1.35
Natural-like (Seen Prots)                  |   2183 |   0.581 |    0.572 |       0.349 |  1.22
Natural-like (NEW Prots / Cold Start)      |    519 |   0.398 |    0.392 |       0.189 |  1.40

 MULTI-SEED RUN (AVERAGED OVER SEEDS [42, 43, 44]): STRICT (NP_5TO9 OR NP_10PLUS)
Test Subset                                | Mean n | Pearson | Spearman | Per-Prot Sp |  RMSE
------------------------------------------------------------------------------------------
Synthetic (Seen Prots, Unsee

In [25]:
# ----------------------------------------------------------------------
# 4. MULTI-SEED EXPERIMENT RUNNER WITH STANDARD DEVIATION
# ----------------------------------------------------------------------
def run_multi_seed(tier_label, is_np_mask, drop_mask=None):
    print(f"\n=========================================================================================")
    print(f" MULTI-SEED RUN WITH STD DEV (3 SEEDS {SEEDS}): {tier_label.upper()}")
    print(f"=========================================================================================")
    
    drop = np.zeros(len(df), bool) if drop_mask is None else drop_mask
    syn_mask = (~is_np_mask) & (~drop)
    np_mask = is_np_mask
    
    set_names = [
        "Synthetic (Seen Prots, Unseen Scaffolds)",
        "Synthetic (NEW Prots / Cold Start)",
        "Natural-like (Seen Prots)",
        "Natural-like (NEW Prots / Cold Start)"
    ]
    
    results = {name: {"pearson": [], "spearman": [], "pp_spearman": [], "rmse": [], "n": []} for name in set_names}
    
    for seed in SEEDS:
        rng = np.random.RandomState(seed)
        
        # 1. Protein split (20% held out)
        all_prots = np.array(sorted(df["uniprot"].unique()))
        rng.shuffle(all_prots)
        test_prots = set(all_prots[:int(len(all_prots) * 0.20)])
        is_new_prot = df["uniprot"].isin(test_prots).values
        
        # 2. Scaffold split for synthetic compounds (20% held out)
        syn_scaffolds = np.array(sorted(df[syn_mask]["scaffold"].unique()))
        rng.shuffle(syn_scaffolds)
        test_scaffolds = set(syn_scaffolds[:int(len(syn_scaffolds) * 0.20)])
        is_new_scaffold = df["scaffold"].isin(test_scaffolds).values
        
        # Training set
        train_mask = syn_mask & (~is_new_prot) & (~is_new_scaffold)
        
        # Test sets
        masks = {
            "Synthetic (Seen Prots, Unseen Scaffolds)": syn_mask & (~is_new_prot) & is_new_scaffold,
            "Synthetic (NEW Prots / Cold Start)": syn_mask & is_new_prot & is_new_scaffold,
            "Natural-like (Seen Prots)": np_mask & (~is_new_prot),
            "Natural-like (NEW Prots / Cold Start)": np_mask & is_new_prot
        }
        
        model = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.08, random_state=seed)
        model.fit(X_all[train_mask], y_all[train_mask])
        y_preds = model.predict(X_all)
        
        for name, m in masks.items():
            pr, sr, ppsr, rmse = eval_mask(m, y_all, y_preds, prot_arr)
            results[name]["pearson"].append(pr)
            results[name]["spearman"].append(sr)
            results[name]["pp_spearman"].append(ppsr)
            results[name]["rmse"].append(rmse)
            results[name]["n"].append(m.sum())

    print(f"{'Test Subset':<42} | {'Mean n':>6} | {'Pearson (Mean ± Std)':>21} | {'Spearman (Mean ± Std)':>22} | {'Per-Prot Sp (Mean ± Std)':>25} | {'RMSE (Mean ± Std)':>17}")
    print("-" * 145)
    for name in set_names:
        n_m = np.mean(results[name]["n"])
        
        pr_m, pr_s = np.nanmean(results[name]["pearson"]), np.nanstd(results[name]["pearson"])
        sr_m, sr_s = np.nanmean(results[name]["spearman"]), np.nanstd(results[name]["spearman"])
        ppsr_m, ppsr_s = np.nanmean(results[name]["pp_spearman"]), np.nanstd(results[name]["pp_spearman"])
        rmse_m, rmse_s = np.nanmean(results[name]["rmse"]), np.nanstd(results[name]["rmse"])
        
        print(f"{name:<42} | {n_m:>6.0f} | {pr_m:6.3f} ± {pr_s:5.3f}         | {sr_m:6.3f} ± {sr_s:5.3f}          | {ppsr_m:6.3f} ± {ppsr_s:5.3f}             | {rmse_m:5.2f} ± {rmse_s:4.2f}")

# ----------------------------------------------------------------------
# 5. EXECUTE BOTH RUNS
# ----------------------------------------------------------------------
# Loose definition: np_tier != "unmatched"
is_np_loose = (df["np_tier"] != "unmatched").values
run_multi_seed("Loose (np_tier != unmatched)", is_np_loose)

# Strict definition: np_tier in ["np_5to9", "np_10plus"], dropping np_1 and np_2to4
is_np_strict = df["np_tier"].isin(["np_5to9", "np_10plus"]).values
drop_mid = df["np_tier"].isin(["np_1", "np_2to4"]).values
run_multi_seed("Strict (np_5to9 or np_10plus)", is_np_strict, drop_mid)


 MULTI-SEED RUN WITH STD DEV (3 SEEDS [42, 43, 44]): LOOSE (NP_TIER != UNMATCHED)
Test Subset                                | Mean n |  Pearson (Mean ± Std) |  Spearman (Mean ± Std) |  Per-Prot Sp (Mean ± Std) | RMSE (Mean ± Std)
-------------------------------------------------------------------------------------------------------------------------------------------------
Synthetic (Seen Prots, Unseen Scaffolds)   |   9448 |  0.712 ± 0.004         |  0.704 ± 0.008          |  0.389 ± 0.033             |  0.93 ± 0.02
Synthetic (NEW Prots / Cold Start)         |   2515 |  0.234 ± 0.084         |  0.226 ± 0.082          |  0.154 ± 0.033             |  1.35 ± 0.09
Natural-like (Seen Prots)                  |   2183 |  0.581 ± 0.011         |  0.572 ± 0.013          |  0.349 ± 0.004             |  1.22 ± 0.01
Natural-like (NEW Prots / Cold Start)      |    519 |  0.398 ± 0.030         |  0.392 ± 0.041          |  0.189 ± 0.011             |  1.40 ± 0.04

 MULTI-SEED RUN WITH STD DEV (3 S

In [28]:
import io
import time
import requests
import copy
import numpy as np
import pandas as pd
from itertools import product
from scipy.stats import pearsonr, spearmanr

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import AllChem

RDLogger.DisableLog("rdApp.*")

# Set computation device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ----------------------------------------------------------------------
# 1. DATA PREPARATION & FILTERING
# ----------------------------------------------------------------------
OUT = "/kaggle/working/final_chembl_np.csv"
SEQ_CSV = "/kaggle/working/sequences.csv"
SEEDS = [42, 43, 44]

df = pd.read_csv(OUT, low_memory=False)

def to_bool(s):
    return s.astype(str).str.strip().str.lower().isin(["true", "1", "1.0"])

mask = (
    to_bool(df["human"]) &
    (df["standard_relation"].astype(str).str.strip() == "=") &
    (df["potential_duplicate"] == 0) &
    df["data_validity_comment"].isna() &
    (df["np_tier"] != "no_structure")
)
df = df[mask].copy()
df["uniprot"] = df["uniprot"].astype(str).str.split(r"[|,;\s]+").str[0]

def calc_pvalue(row):
    p = row.get("pchembl_value")
    if pd.notna(p):
        try:
            return float(p)
        except ValueError:
            pass
    val = row.get("standard_value")
    unit = str(row.get("standard_units")).strip().lower()
    if pd.notna(val) and unit in ["nm", "nanomolar"]:
        try:
            v = float(val)
            if v > 0:
                return 9.0 - np.log10(v)
        except ValueError:
            pass
    return np.nan

df["pvalue"] = [calc_pvalue(row) for _, row in df.iterrows()]
df = df.dropna(subset=["pvalue", "canonical_smiles", "uniprot"]).copy()

# Deduplicate to median pvalue per pair
df = (
    df.groupby(["canonical_smiles", "uniprot"], as_index=False)
      .agg({"pvalue": "median", "np_tier": "first", "doc_id": "first"})
)

# ----------------------------------------------------------------------
# 2. SEQUENCES & FEATURE EXTRACTION
# ----------------------------------------------------------------------
accs = sorted(df["uniprot"].unique().tolist())
try:
    seq_df = pd.read_csv(SEQ_CSV)
    missing_accs = [a for a in accs if a not in seq_df["uniprot"].values]
except FileNotFoundError:
    seq_df = pd.DataFrame(columns=["uniprot", "sequence"])
    missing_accs = accs

if missing_accs:
    parts = []
    for i in range(0, len(missing_accs), 100):
        chunk = missing_accs[i:i+100]
        for attempt in range(3):
            r = requests.get(
                "https://rest.uniprot.org/uniprotkb/accessions",
                params={"accessions": ",".join(chunk), "fields": "accession,sequence", "format": "tsv"},
                timeout=60
            )
            if r.status_code == 200:
                break
            time.sleep(2)
        else:
            continue
        t = pd.read_csv(io.StringIO(r.text), sep="\t", dtype=str)
        t.columns = ["uniprot", "sequence"]
        parts.append(t)
    if parts:
        new_seqs = pd.concat(parts, ignore_index=True).drop_duplicates("uniprot")
        seq_df = pd.concat([seq_df, new_seqs], ignore_index=True).drop_duplicates("uniprot")
        seq_df.to_csv(SEQ_CSV, index=False)

seq_dict = seq_df.set_index("uniprot")["sequence"].to_dict()
df = df[df["uniprot"].isin(seq_dict.keys())].copy()

AA = "ACDEFGHIKLMNPQRSTVWY"
DIPEP = {a + b: i for i, (a, b) in enumerate(product(AA, AA))}

def get_prot_feats(seq):
    seq = "".join(c for c in str(seq).upper() if c in AA)
    comp, dip = np.zeros(20), np.zeros(400)
    if len(seq) > 1:
        for c in seq: comp[AA.index(c)] += 1
        comp /= len(seq)
        for i in range(len(seq) - 1): dip[DIPEP[seq[i:i + 2]]] += 1
        dip /= len(seq) - 1
    return np.concatenate([comp, dip, [np.log1p(len(seq))]])

def get_fp(smi, n=2048):
    m = Chem.MolFromSmiles(smi)
    if m is None: return None
    v = AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=n)
    return np.array(list(v), dtype=np.uint8)

def get_scaffold(smi):
    m = Chem.MolFromSmiles(smi)
    if m is None: return smi
    try:
        scaf = MurckoScaffold.MurckoScaffoldSmiles(mol=m)
        return scaf if scaf else smi
    except Exception:
        return smi

unique_smiles = df["canonical_smiles"].unique()
fp_dict = {s: get_fp(s) for s in unique_smiles}
scaf_dict = {s: get_scaffold(s) for s in unique_smiles}

df = df[df["canonical_smiles"].map(lambda s: fp_dict[s] is not None)].copy()
df["scaffold"] = df["canonical_smiles"].map(scaf_dict)

unique_prots = df["uniprot"].unique()
pf_dict = {u: get_prot_feats(seq_dict[u]) for u in unique_prots}

X_fps = np.stack(df["canonical_smiles"].map(fp_dict).values)
X_pfs = np.stack(df["uniprot"].map(pf_dict).values)
X_all = np.hstack([X_fps, X_pfs]).astype(np.float32)
y_all = df["pvalue"].values.astype(np.float32)
prot_arr = df["uniprot"].values

is_np_loose = (df["np_tier"] != "unmatched").values
print(f"Total pairs: {len(df):,} | Features: {X_all.shape[1]}")

# ----------------------------------------------------------------------
# 3. PYTORCH MLP ARCHITECTURE
# ----------------------------------------------------------------------
class TransferMLP(nn.Module):
    def __init__(self, in_features=2461):
        super(TransferMLP, self).__init__()
        self.fc1 = nn.Linear(in_features, 1024)
        self.bn1 = nn.BatchNorm1d(1024)
        self.relu1 = nn.ReLU()
        
        self.fc2 = nn.Linear(1024, 256)
        self.bn2 = nn.BatchNorm1d(256)
        self.relu2 = nn.ReLU()
        
        self.out = nn.Linear(256, 1)

    def forward(self, x):
        x = self.relu1(self.bn1(self.fc1(x)))
        x = self.relu2(self.bn2(self.fc2(x)))
        x = self.out(x)
        return x.squeeze(-1)

def train_model(model, dataloader, epochs=20, lr=1e-3, freeze_hidden=False):
    model = model.to(device)
    
    if freeze_hidden:
        for param in model.fc1.parameters(): param.requires_grad = False
        for param in model.fc2.parameters(): param.requires_grad = False
        params = list(model.bn1.parameters()) + list(model.bn2.parameters()) + list(model.out.parameters())
    else:
        for param in model.parameters(): param.requires_grad = True
        params = model.parameters()
        
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    criterion = nn.MSELoss()
    
    model.train()
    for epoch in range(epochs):
        for bx, by in dataloader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            pred = model(bx)
            loss = criterion(pred, by)
            loss.backward()
            optimizer.step()
    return model

def predict(model, X_matrix, batch_size=512):
    model.eval()
    dataset = TensorDataset(torch.from_numpy(X_matrix))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    preds = []
    with torch.no_grad():
        for (bx,) in loader:
            bx = bx.to(device)
            p = model(bx)
            preds.append(p.cpu().numpy())
    return np.concatenate(preds)

# ----------------------------------------------------------------------
# 4. METRICS HELPERS
# ----------------------------------------------------------------------
def per_protein_spearman(y_true, y_pred, prots, min_samples=5):
    eval_df = pd.DataFrame({"y_true": y_true, "y_pred": y_pred, "prot": prots})
    spearmans = []
    for _, group in eval_df.groupby("prot"):
        if len(group) >= min_samples and group["y_true"].nunique() > 1:
            s = spearmanr(group["y_true"], group["y_pred"])[0]
            if not np.isnan(s):
                spearmans.append(s)
    return np.mean(spearmans) if spearmans else np.nan

def eval_performance(mask, y_true, y_pred, prots):
    if mask.sum() < 10:
        return np.nan, np.nan, np.nan, np.nan
    yt, yp, pr = y_true[mask], y_pred[mask], prots[mask]
    p_r = pearsonr(yt, yp)[0]
    s_r = spearmanr(yt, yp)[0]
    rmse = np.sqrt(np.mean((yt - yp) ** 2))
    pp_sp = per_protein_spearman(yt, yp, pr, min_samples=5)
    return p_r, s_r, pp_sp, rmse

# ----------------------------------------------------------------------
# 5. MULTI-SEED PIPELINE
# ----------------------------------------------------------------------
ARM1 = "Arm 1: Synthetic Zero-Shot (Natural Test)"
ARM2 = "Arm 2: Full Fine-Tuning (Natural Test)"
ARM3 = "Arm 3: Head + BN Fine-Tuning (Natural Test)"
ARM4 = "Arm 4: Natural-only from Scratch (Natural Test)"
ARM2_FORGET = "Arm 2: Synthetic Test (Catastrophic Forgetting Check)"

arm_names = [ARM1, ARM2, ARM3, ARM4, ARM2_FORGET]
results = {name: {"pearson": [], "spearman": [], "pp_spearman": [], "rmse": [], "n": []} for name in arm_names}

for seed in SEEDS:
    rng = np.random.RandomState(seed)
    torch.manual_seed(seed)
    
    syn_mask = ~is_np_loose
    np_mask = is_np_loose
    
    # 1. Protein split (20% held out)
    all_prots = np.array(sorted(df["uniprot"].unique()))
    rng.shuffle(all_prots)
    test_prots = set(all_prots[:int(len(all_prots) * 0.20)])
    is_new_prot = df["uniprot"].isin(test_prots).values
    
    # 2. Synthetic Scaffold split (20% held out)
    syn_scaffolds = np.array(sorted(df[syn_mask]["scaffold"].unique()))
    rng.shuffle(syn_scaffolds)
    test_syn_scaffolds = set(syn_scaffolds[:int(len(syn_scaffolds) * 0.20)])
    is_new_syn_scaffold = df["scaffold"].isin(test_syn_scaffolds).values
    
    train_syn_mask = syn_mask & (~is_new_prot) & (~is_new_syn_scaffold)
    test_syn_mask = syn_mask & (~is_new_prot) & is_new_syn_scaffold
    
    # 3. Natural Scaffold split (70% Fine-Tune / 30% Test)
    np_scaffolds = np.array(sorted(df[np_mask]["scaffold"].unique()))
    rng.shuffle(np_scaffolds)
    n_ft_scaffolds = int(len(np_scaffolds) * 0.70)
    ft_np_scaffolds = set(np_scaffolds[:n_ft_scaffolds])
    is_ft_np_scaffold = df["scaffold"].isin(ft_np_scaffolds).values
    
    finetune_np_mask = np_mask & (~is_new_prot) & is_ft_np_scaffold
    test_np_mask = np_mask & (~is_new_prot) & (~is_ft_np_scaffold)
    
    if seed == SEEDS[0]:
        print("\n=======================================================")
        print(" NATURAL-LIKE ROW SPLIT COUNTS (LOOSE LABEL)")
        print("=======================================================")
        print(f"Natural Fine-Tune Rows (70% Scaffolds): {finetune_np_mask.sum():,}")
        print(f"Natural Test Rows      (30% Scaffolds): {test_np_mask.sum():,}")
        print("=======================================================\n")
    
    # --- ARM 1 ---
    ds_syn = TensorDataset(torch.from_numpy(X_all[train_syn_mask]), torch.from_numpy(y_all[train_syn_mask]))
    loader_syn = DataLoader(ds_syn, batch_size=256, shuffle=True)
    base_model = TransferMLP(in_features=X_all.shape[1])
    base_model = train_model(base_model, loader_syn, epochs=25, lr=1e-3, freeze_hidden=False)
    
    preds_arm1 = predict(base_model, X_all)
    print("Arm 1 on synthetic test, RMSE:", eval_performance(test_syn_mask, y_all, preds_arm1, prot_arr)[3])
    pr, sr, ppsr, rmse = eval_performance(test_np_mask, y_all, preds_arm1, prot_arr)
    results[ARM1]["pearson"].append(pr)
    results[ARM1]["spearman"].append(sr)
    results[ARM1]["pp_spearman"].append(ppsr)
    results[ARM1]["rmse"].append(rmse)
    results[ARM1]["n"].append(test_np_mask.sum())
    
    ds_np_ft = TensorDataset(torch.from_numpy(X_all[finetune_np_mask]), torch.from_numpy(y_all[finetune_np_mask]))
    loader_np_ft = DataLoader(ds_np_ft, batch_size=64, shuffle=True)
    
    # --- ARM 2 ---
    model_arm2 = copy.deepcopy(base_model)
    model_arm2 = train_model(model_arm2, loader_np_ft, epochs=15, lr=1e-4, freeze_hidden=False)
    preds_arm2 = predict(model_arm2, X_all)
    
    pr, sr, ppsr, rmse = eval_performance(test_np_mask, y_all, preds_arm2, prot_arr)
    results[ARM2]["pearson"].append(pr)
    results[ARM2]["spearman"].append(sr)
    results[ARM2]["pp_spearman"].append(ppsr)
    results[ARM2]["rmse"].append(rmse)
    results[ARM2]["n"].append(test_np_mask.sum())
    
    # Forgetting check
    pr, sr, ppsr, rmse = eval_performance(test_syn_mask, y_all, preds_arm2, prot_arr)
    results[ARM2_FORGET]["pearson"].append(pr)
    results[ARM2_FORGET]["spearman"].append(sr)
    results[ARM2_FORGET]["pp_spearman"].append(ppsr)
    results[ARM2_FORGET]["rmse"].append(rmse)
    results[ARM2_FORGET]["n"].append(test_syn_mask.sum())
    
    # --- ARM 3 ---
    model_arm3 = copy.deepcopy(base_model)
    model_arm3 = train_model(model_arm3, loader_np_ft, epochs=15, lr=1e-4, freeze_hidden=True)
    preds_arm3 = predict(model_arm3, X_all)
    
    pr, sr, ppsr, rmse = eval_performance(test_np_mask, y_all, preds_arm3, prot_arr)
    results[ARM3]["pearson"].append(pr)
    results[ARM3]["spearman"].append(sr)
    results[ARM3]["pp_spearman"].append(ppsr)
    results[ARM3]["rmse"].append(rmse)
    results[ARM3]["n"].append(test_np_mask.sum())
    
    # --- ARM 4 ---
    model_arm4 = TransferMLP(in_features=X_all.shape[1])
    model_arm4 = train_model(model_arm4, loader_np_ft, epochs=30, lr=1e-3, freeze_hidden=False)
    preds_arm4 = predict(model_arm4, X_all)
    
    pr, sr, ppsr, rmse = eval_performance(test_np_mask, y_all, preds_arm4, prot_arr)
    results[ARM4]["pearson"].append(pr)
    results[ARM4]["spearman"].append(sr)
    results[ARM4]["pp_spearman"].append(ppsr)
    results[ARM4]["rmse"].append(rmse)
    results[ARM4]["n"].append(test_np_mask.sum())

# ----------------------------------------------------------------------
# 6. RESULTS OUTPUT
# ----------------------------------------------------------------------
print(f"=========================================================================================================")
print(f" PYTORCH MLP TRANSFER EXPERIMENT (AVERAGED OVER SEEDS {SEEDS})")
print(f"=========================================================================================================")
print(f"{'Experiment Arm':<55} | {'Mean n':>6} | {'Per-Prot Sp (Mean ± Std)':>25} | {'RMSE (Mean ± Std)':>17}")
print("-" * 115)

for name in arm_names:
    n_m = np.mean(results[name]["n"])
    ppsr_m, ppsr_s = np.nanmean(results[name]["pp_spearman"]), np.nanstd(results[name]["pp_spearman"])
    rmse_m, rmse_s = np.nanmean(results[name]["rmse"]), np.nanstd(results[name]["rmse"])
    print(f"{name:<55} | {n_m:>6.0f} | {ppsr_m:6.3f} ± {ppsr_s:5.3f}             | {rmse_m:5.2f} ± {rmse_s:4.2f}")

Using device: cpu
Total pairs: 62,906 | Features: 2469

 NATURAL-LIKE ROW SPLIT COUNTS (LOOSE LABEL)
Natural Fine-Tune Rows (70% Scaffolds): 1,618
Natural Test Rows      (30% Scaffolds): 532

Arm 1 on synthetic test, RMSE: 0.95265543
Arm 1 on synthetic test, RMSE: 0.9181438
Arm 1 on synthetic test, RMSE: 0.91841453
 PYTORCH MLP TRANSFER EXPERIMENT (AVERAGED OVER SEEDS [42, 43, 44])
Experiment Arm                                          | Mean n |  Per-Prot Sp (Mean ± Std) | RMSE (Mean ± Std)
-------------------------------------------------------------------------------------------------------------------
Arm 1: Synthetic Zero-Shot (Natural Test)               |    603 |  0.351 ± 0.056             |  1.29 ± 0.02
Arm 2: Full Fine-Tuning (Natural Test)                  |    603 |  0.343 ± 0.015             |  1.28 ± 0.02
Arm 3: Head + BN Fine-Tuning (Natural Test)             |    603 |  0.344 ± 0.051             |  1.28 ± 0.01
Arm 4: Natural-only from Scratch (Natural Test)         |  

In [30]:
import io
import time
import requests
import copy
import numpy as np
import pandas as pd
from itertools import product
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import AllChem

RDLogger.DisableLog("rdApp.*")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ----------------------------------------------------------------------
# 1. DATA PREPARATION & FILTERING
# ----------------------------------------------------------------------
OUT = "/kaggle/working/final_chembl_np.csv"
SEQ_CSV = "/kaggle/working/sequences.csv"
SEEDS = [42, 43, 44]
FRACTIONS = [0.25, 0.50, 1.00]

df = pd.read_csv(OUT, low_memory=False)

def to_bool(s):
    return s.astype(str).str.strip().str.lower().isin(["true", "1", "1.0"])

mask = (
    to_bool(df["human"]) &
    (df["standard_relation"].astype(str).str.strip() == "=") &
    (df["potential_duplicate"] == 0) &
    df["data_validity_comment"].isna() &
    (df["np_tier"] != "no_structure")
)
df = df[mask].copy()
df["uniprot"] = df["uniprot"].astype(str).str.split(r"[|,;\s]+").str[0]

def calc_pvalue(row):
    p = row.get("pchembl_value")
    if pd.notna(p):
        try:
            return float(p)
        except ValueError:
            pass
    val = row.get("standard_value")
    unit = str(row.get("standard_units")).strip().lower()
    if pd.notna(val) and unit in ["nm", "nanomolar"]:
        try:
            v = float(val)
            if v > 0:
                return 9.0 - np.log10(v)
        except ValueError:
            pass
    return np.nan

df["pvalue"] = [calc_pvalue(row) for _, row in df.iterrows()]
df = df.dropna(subset=["pvalue", "canonical_smiles", "uniprot"]).copy()

df = (
    df.groupby(["canonical_smiles", "uniprot"], as_index=False)
      .agg({"pvalue": "median", "np_tier": "first", "doc_id": "first"})
)

# ----------------------------------------------------------------------
# 2. SEQUENCES & FEATURE EXTRACTION
# ----------------------------------------------------------------------
accs = sorted(df["uniprot"].unique().tolist())
try:
    seq_df = pd.read_csv(SEQ_CSV)
    missing_accs = [a for a in accs if a not in seq_df["uniprot"].values]
except FileNotFoundError:
    seq_df = pd.DataFrame(columns=["uniprot", "sequence"])
    missing_accs = accs

if missing_accs:
    parts = []
    for i in range(0, len(missing_accs), 100):
        chunk = missing_accs[i:i+100]
        for attempt in range(3):
            r = requests.get(
                "https://rest.uniprot.org/uniprotkb/accessions",
                params={"accessions": ",".join(chunk), "fields": "accession,sequence", "format": "tsv"},
                timeout=60
            )
            if r.status_code == 200:
                break
            time.sleep(2)
        else:
            continue
        t = pd.read_csv(io.StringIO(r.text), sep="\t", dtype=str)
        t.columns = ["uniprot", "sequence"]
        parts.append(t)
    if parts:
        new_seqs = pd.concat(parts, ignore_index=True).drop_duplicates("uniprot")
        seq_df = pd.concat([seq_df, new_seqs], ignore_index=True).drop_duplicates("uniprot")
        seq_df.to_csv(SEQ_CSV, index=False)

seq_dict = seq_df.set_index("uniprot")["sequence"].to_dict()
df = df[df["uniprot"].isin(seq_dict.keys())].copy()

AA = "ACDEFGHIKLMNPQRSTVWY"
DIPEP = {a + b: i for i, (a, b) in enumerate(product(AA, AA))}

def get_prot_feats(seq):
    seq = "".join(c for c in str(seq).upper() if c in AA)
    comp, dip = np.zeros(20), np.zeros(400)
    if len(seq) > 1:
        for c in seq: comp[AA.index(c)] += 1
        comp /= len(seq)
        for i in range(len(seq) - 1): dip[DIPEP[seq[i:i + 2]]] += 1
        dip /= len(seq) - 1
    return np.concatenate([comp, dip, [np.log1p(len(seq))]])

def get_fp(smi, n=2048):
    m = Chem.MolFromSmiles(smi)
    if m is None: return None
    v = AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=n)
    return np.array(list(v), dtype=np.uint8)

def get_scaffold(smi):
    m = Chem.MolFromSmiles(smi)
    if m is None: return smi
    try:
        scaf = MurckoScaffold.MurckoScaffoldSmiles(mol=m)
        return scaf if scaf else smi
    except Exception:
        return smi

unique_smiles = df["canonical_smiles"].unique()
fp_dict = {s: get_fp(s) for s in unique_smiles}
scaf_dict = {s: get_scaffold(s) for s in unique_smiles}

df = df[df["canonical_smiles"].map(lambda s: fp_dict[s] is not None)].copy()
df["scaffold"] = df["canonical_smiles"].map(scaf_dict)

unique_prots = df["uniprot"].unique()
pf_dict = {u: get_prot_feats(seq_dict[u]) for u in unique_prots}

X_fps = np.stack(df["canonical_smiles"].map(fp_dict).values)
X_pfs = np.stack(df["uniprot"].map(pf_dict).values)
X_all = np.hstack([X_fps, X_pfs]).astype(np.float32)
y_all = df["pvalue"].values.astype(np.float32)
prot_arr = df["uniprot"].values

is_np_loose = (df["np_tier"] != "unmatched").values

# ----------------------------------------------------------------------
# 3. PYTORCH MLP ARCHITECTURE
# ----------------------------------------------------------------------
class TransferMLP(nn.Module):
    def __init__(self, in_features=2461):
        super(TransferMLP, self).__init__()
        self.fc1 = nn.Linear(in_features, 1024)
        self.bn1 = nn.BatchNorm1d(1024)
        self.relu1 = nn.ReLU()
        
        self.fc2 = nn.Linear(1024, 256)
        self.bn2 = nn.BatchNorm1d(256)
        self.relu2 = nn.ReLU()
        
        self.out = nn.Linear(256, 1)

    def forward(self, x):
        x = self.relu1(self.bn1(self.fc1(x)))
        x = self.relu2(self.bn2(self.fc2(x)))
        x = self.out(x)
        return x.squeeze(-1)

def train_model(model, dataloader, epochs=20, lr=1e-3):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.MSELoss()
    
    model.train()
    for epoch in range(epochs):
        for bx, by in dataloader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            pred = model(bx)
            loss = criterion(pred, by)
            loss.backward()
            optimizer.step()
    return model

def predict(model, X_matrix, batch_size=512):
    model.eval()
    dataset = TensorDataset(torch.from_numpy(X_matrix))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    preds = []
    with torch.no_grad():
        for (bx,) in loader:
            bx = bx.to(device)
            p = model(bx)
            preds.append(p.cpu().numpy())
    return np.concatenate(preds)

# ----------------------------------------------------------------------
# 4. METRICS HELPERS
# ----------------------------------------------------------------------
def per_protein_spearman(y_true, y_pred, prots, min_samples=5):
    eval_df = pd.DataFrame({"y_true": y_true, "y_pred": y_pred, "prot": prots})
    spearmans = []
    for _, group in eval_df.groupby("prot"):
        if len(group) >= min_samples and group["y_true"].nunique() > 1:
            s = spearmanr(group["y_true"], group["y_pred"])[0]
            if not np.isnan(s):
                spearmans.append(s)
    return np.mean(spearmans) if spearmans else np.nan

def eval_performance(mask, y_true, y_pred, prots):
    if mask.sum() < 10:
        return np.nan, np.nan, np.nan, np.nan
    yt, yp, pr = y_true[mask], y_pred[mask], prots[mask]
    p_r = pearsonr(yt, yp)[0]
    s_r = spearmanr(yt, yp)[0]
    rmse = np.sqrt(np.mean((yt - yp) ** 2))
    pp_sp = per_protein_spearman(yt, yp, pr, min_samples=5)
    return p_r, s_r, pp_sp, rmse

# ----------------------------------------------------------------------
# 5. LEARNING-CURVE EXPERIMENT PIPELINE WITH 0% POINT
# ----------------------------------------------------------------------
zero_results = {"rmse": [], "pp_spearman": []}
results = {f: {"rmse": [], "pp_spearman": [], "n_ft": []} for f in FRACTIONS}

for seed in SEEDS:
    rng = np.random.RandomState(seed)
    torch.manual_seed(seed)
    
    syn_mask = ~is_np_loose
    np_mask = is_np_loose
    
    # 1. Protein split (20% held out)
    all_prots = np.array(sorted(df["uniprot"].unique()))
    rng.shuffle(all_prots)
    test_prots = set(all_prots[:int(len(all_prots) * 0.20)])
    is_new_prot = df["uniprot"].isin(test_prots).values
    
    # 2. Synthetic Scaffold split (20% held out)
    syn_scaffolds = np.array(sorted(df[syn_mask]["scaffold"].unique()))
    rng.shuffle(syn_scaffolds)
    test_syn_scaffolds = set(syn_scaffolds[:int(len(syn_scaffolds) * 0.20)])
    is_new_syn_scaffold = df["scaffold"].isin(test_syn_scaffolds).values
    
    train_syn_mask = syn_mask & (~is_new_prot) & (~is_new_syn_scaffold)
    
    # 3. Natural Scaffold split (70% Fine-Tune / 30% Test)
    np_scaffolds = np.array(sorted(df[np_mask]["scaffold"].unique()))
    rng.shuffle(np_scaffolds)
    n_ft_scaffolds = int(len(np_scaffolds) * 0.70)
    ft_np_scaffolds = set(np_scaffolds[:n_ft_scaffolds])
    is_ft_np_scaffold = df["scaffold"].isin(ft_np_scaffolds).values
    
    full_finetune_mask = np_mask & (~is_new_prot) & is_ft_np_scaffold
    test_np_mask = np_mask & (~is_new_prot) & (~is_ft_np_scaffold)
    
    # Train base synthetic model once per seed
    ds_syn = TensorDataset(torch.from_numpy(X_all[train_syn_mask]), torch.from_numpy(y_all[train_syn_mask]))
    loader_syn = DataLoader(ds_syn, batch_size=256, shuffle=True, drop_last=True)
    base_model = TransferMLP(in_features=X_all.shape[1])
    base_model = train_model(base_model, loader_syn, epochs=25, lr=1e-3)
    
    # Evaluate 0% baseline (Zero-Shot synthetic model directly on natural test)
    preds_zero = predict(base_model, X_all)
    _, _, pp0, r0 = eval_performance(test_np_mask, y_all, preds_zero, prot_arr)
    zero_results["rmse"].append(r0)
    zero_results["pp_spearman"].append(pp0)
    
    ft_indices = np.where(full_finetune_mask)[0]
    
    for f in FRACTIONS:
        n_sub = int(len(ft_indices) * f)
        sub_indices = rng.choice(ft_indices, size=n_sub, replace=False)
        sub_mask = np.zeros(len(df), dtype=bool)
        sub_mask[sub_indices] = True
        
        # drop_last=True avoids batch size = 1 errors in BatchNorm1d
        ds_sub = TensorDataset(torch.from_numpy(X_all[sub_mask]), torch.from_numpy(y_all[sub_mask]))
        loader_sub = DataLoader(ds_sub, batch_size=64, shuffle=True, drop_last=True)
        
        # Fine-tune base model copy on fraction f
        model_sub = copy.deepcopy(base_model)
        model_sub = train_model(model_sub, loader_sub, epochs=15, lr=1e-4)
        
        preds_sub = predict(model_sub, X_all)
        _, _, ppsr, rmse = eval_performance(test_np_mask, y_all, preds_sub, prot_arr)
        
        results[f]["rmse"].append(rmse)
        results[f]["pp_spearman"].append(ppsr)
        results[f]["n_ft"].append(n_sub)

# ----------------------------------------------------------------------
# 6. DISPLAY RESULTS TABLE & SAVE PLOT
# ----------------------------------------------------------------------
print("\n=========================================================================================")
print(f" FINE-TUNING LEARNING-CURVE RESULTS (AVERAGED OVER SEEDS {SEEDS})")
print("=========================================================================================")
print(f"{'Data Fraction (%)':<20} | {'Mean FT Rows (n)':>16} | {'Per-Prot Sp (Mean ± Std)':>25} | {'RMSE (Mean ± Std)':>17}")
print("-" * 88)

# Print 0% anchor point
r0_m, r0_s = np.nanmean(zero_results["rmse"]), np.nanstd(zero_results["rmse"])
pp0_m, pp0_s = np.nanmean(zero_results["pp_spearman"]), np.nanstd(zero_results["pp_spearman"])
print(f"  0% (No Fine-Tuning)  | {0:>16d} | {pp0_m:6.3f} ± {pp0_s:5.3f}             | {r0_m:5.2f} ± {r0_s:4.2f}")

# Print fine-tuned fractions
for f in FRACTIONS:
    pct = int(f * 100)
    n_m = np.mean(results[f]["n_ft"])
    ppsr_m, ppsr_s = np.nanmean(results[f]["pp_spearman"]), np.nanstd(results[f]["pp_spearman"])
    rmse_m, rmse_s = np.nanmean(results[f]["rmse"]), np.nanstd(results[f]["rmse"])
    print(f"{pct:>3}%                  | {n_m:>16.0f} | {ppsr_m:6.3f} ± {ppsr_s:5.3f}             | {rmse_m:5.2f} ± {rmse_s:4.2f}")

# Plot and save learning curve including 0%
fractions_pct = [0] + [int(f * 100) for f in FRACTIONS]
rmse_means = [r0_m] + [np.nanmean(results[f]["rmse"]) for f in FRACTIONS]
rmse_stds = [r0_s] + [np.nanstd(results[f]["rmse"]) for f in FRACTIONS]

plt.figure(figsize=(7, 4.5))
plt.errorbar(fractions_pct, rmse_means, yerr=rmse_stds, fmt='-o', capsize=5, color='#1f77b4', linewidth=2, label='Natural Test RMSE')
plt.title('Fine-Tuning Learning Curve: RMSE vs Natural Data Size')
plt.xlabel('Percentage of Natural Fine-Tuning Data (%)')
plt.ylabel('RMSE (Lower is Better)')
plt.xticks(fractions_pct)
plt.grid(True, linestyle='--', alpha=0.6)
plt.savefig('/kaggle/working/learning_curve.png', dpi=300, bbox_inches='tight')
plt.close()

print("\nSaved learning curve plot to /kaggle/working/learning_curve.png")

Using device: cpu

 FINE-TUNING LEARNING-CURVE RESULTS (AVERAGED OVER SEEDS [42, 43, 44])
Data Fraction (%)    | Mean FT Rows (n) |  Per-Prot Sp (Mean ± Std) | RMSE (Mean ± Std)
----------------------------------------------------------------------------------------
  0% (No Fine-Tuning)  |                0 |  0.374 ± 0.039             |  1.29 ± 0.01
 25%                  |              395 |  0.352 ± 0.005             |  1.32 ± 0.03
 50%                  |              790 |  0.364 ± 0.004             |  1.28 ± 0.03
100%                  |             1580 |  0.366 ± 0.012             |  1.28 ± 0.02

Saved learning curve plot to /kaggle/working/learning_curve.png


In [31]:
import os
import zipfile

# Files generated in your working directory
files_to_backup = [
    "final_chembl_np.csv",
    "sequences.csv",
    "modeling_data.csv",
    "learning_curve.png"
]

output_zip = "/kaggle/working/project_backup.zip"

with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as zipf:
    for filename in files_to_backup:
        filepath = os.path.join("/kaggle/working", filename)
        if os.path.exists(filepath):
            zipf.write(filepath, arcname=filename)
            print(f"Added to archive: {filename}")
        else:
            print(f"Skipped (not found): {filename}")

print(f"\nArchive created successfully: {output_zip}")

Added to archive: final_chembl_np.csv
Added to archive: sequences.csv
Skipped (not found): modeling_data.csv
Added to archive: learning_curve.png

Archive created successfully: /kaggle/working/project_backup.zip


In [33]:
import os

working_dir = "/kaggle/working"
files = os.listdir(working_dir)

print(f"{'Filename':<35} | {'Size (MB)':>10}")
print("-" * 48)
for f in sorted(files):
    path = os.path.join(working_dir, f)
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"{f:<35} | {size_mb:>10.2f} MB")

Filename                            |  Size (MB)
------------------------------------------------
baseline_results.csv                |       0.00 MB
coconut_lookup.csv                  |      11.07 MB
family_summary.csv                  |       0.00 MB
family_summary1.csv                 |       0.00 MB
family_summary3.csv                 |       0.00 MB
final_chembl_np.csv                 |      42.16 MB
learning_curve.png                  |       0.10 MB
project_backup.zip                  |       8.23 MB
protein_family_lookup.csv           |       1.58 MB
sequences.csv                       |       0.40 MB


In [35]:
import os

files_to_check = [
    "baseline_results.csv",
    "family_summary.csv",
    "family_summary1.csv",
    "family_summary3.csv",
]

print(f"{'Filename':<25} | {'Bytes':>8} | {'Line Count':>10}")
print("-" * 50)
for fname in files_to_check:
    path = os.path.join("/kaggle/working", fname)
    if os.path.exists(path):
        size_bytes = os.path.getsize(path)
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            lines = sum(1 for _ in f)
        print(f"{fname:<25} | {size_bytes:>8} B | {lines:>10} lines")

Filename                  |    Bytes | Line Count
--------------------------------------------------
baseline_results.csv      |     1251 B |         13 lines
family_summary.csv        |     1936 B |         25 lines
family_summary1.csv       |     1936 B |         25 lines
family_summary3.csv       |     1936 B |         25 lines


In [37]:
import os
import zipfile

working_dir = "/kaggle/working"
output_zip = "/kaggle/working/full_working_backup.zip"

# Get all files in working directory excluding the output zip file itself
files = [
    f for f in os.listdir(working_dir)
    if os.path.isfile(os.path.join(working_dir, f)) and f != "full_working_backup.zip"
]

with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as zipf:
    for filename in sorted(files):
        filepath = os.path.join(working_dir, filename)
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        zipf.write(filepath, arcname=filename)
        print(f"Zipped: {filename:<35} ({size_mb:>7.2f} MB)")

print(f"\nArchive created: {output_zip}")

Zipped: baseline_results.csv                (   0.00 MB)
Zipped: coconut_lookup.csv                  (  11.07 MB)
Zipped: family_summary.csv                  (   0.00 MB)
Zipped: family_summary1.csv                 (   0.00 MB)
Zipped: family_summary3.csv                 (   0.00 MB)
Zipped: final_chembl_np.csv                 (  42.16 MB)
Zipped: learning_curve.png                  (   0.10 MB)
Zipped: project_backup.zip                  (   8.23 MB)
Zipped: protein_family_lookup.csv           (   1.58 MB)
Zipped: sequences.csv                       (   0.40 MB)

Archive created: /kaggle/working/full_working_backup.zip
